# TESS Variable Star Categorization

## A Reproducible Research Notebook on Photometric Provenance and Variable-Star Classification

This notebook is written as a **paper-style research document**. The main text emphasizes scientific motivation, methodology, results, and interpretation. Full implementation code is included for reproducibility, but long code cells are collapsed by default so the notebook remains readable.

The central research question is:

> **How does photometric provenance influence the characterization and classification of variable stars observed by TESS?**

# Chapter 1 – Scientific Motivation and Research Question

## 1.1 Motivation

Variable stars are central objects in observational astrophysics because their brightness variations encode information about stellar interiors, pulsation, binary geometry, rotation, accretion, circumstellar material, and stellar evolution. RR Lyrae stars and Cepheids are especially important because their pulsations are tied to distance-scale work, while eclipsing binaries provide geometric constraints on stellar radii and masses.

The **Transiting Exoplanet Survey Satellite (TESS)** has become a major resource for time-domain stellar astrophysics because it provides high-cadence, space-based photometry across most of the sky. However, TESS light curves are not produced by a single uniform pipeline. In this project, the main photometric provenance categories are:

- **SPOC**: Science Processing Operations Center light curves.
- **QLP**: Quick-Look Pipeline light curves.
- **TESSCut**: light curves extracted from Full-Frame Image cutouts.

These products observe the same underlying sky but differ in extraction, calibration, aperture selection, background treatment, and systematic correction. Therefore, the same star may have different measured amplitude, scatter, morphology, or periodogram structure depending on the photometric product used.

This project treats that difference not as a technical nuisance, but as the central scientific subject.

## 1.2 Primary Research Question

> **How does photometric provenance influence the characterization and classification of variable stars observed by TESS?**

The goal is not simply to maximize Random Forest accuracy. Instead, machine learning is used as a **diagnostic tool** to reveal how different TESS photometric products alter measurable variability signatures.

Supporting questions:

1. How do SPOC, QLP, and TESSCut differ in availability across variable-star families?
2. How do measured period, amplitude, scatter, and morphology features change with provenance?
3. Which variable-star classes are most sensitive to provenance?
4. How does provenance affect Random Forest classification accuracy and feature importance?
5. Can phase-folded morphology explain provenance-dependent classification behavior?
6. Are observed provenance effects likely caused by real photometric-product differences, or could they be explained by VSX-to-TIC cross-match errors?

## 1.3 Research Gap

Prior studies have applied machine learning to classify variable stars using Kepler and TESS light curves. However, the reviewed literature does not appear to systematically quantify how different TESS light-curve products affect downstream variable-star classification, feature importance, and class-specific astrophysical interpretation.

This project therefore focuses on **photometric provenance** as the scientific variable of interest.

A stronger framing of the project is not:

> Random Forest classification of TESS variable stars.

Rather, it is:

> A study of how different TESS photometric products alter the measured variability signatures of astrophysical classes, using machine learning as one diagnostic tool.

# Chapter 2 – Dataset Construction

## 2.1 Overview of the Dataset Pipeline

The dataset-construction pipeline transforms a catalog of known variable stars into a curated multi-provenance TESS light-curve dataset.

```text
AAVSO VSX variable-star catalog
        ↓
Normalize detailed VSX types into broader astrophysical families
        ↓
Cross-match each VSX object to nearby TIC candidates
        ↓
Retain up to five TIC candidates within 5.0 arcsec
        ↓
Try SPOC light curve
        ↓ if unavailable
Try QLP light curve
        ↓ if unavailable
Extract TESSCut FFI photometry
        ↓
Save raw and standardized light curves
        ↓
FITS sanity checks and metadata quality control
        ↓
Final dataset for feature extraction and provenance analysis
```

**Detrending is not performed during this dataset-construction phase.** Detrending is evaluated later as a separate scientific experiment.

In [ ]:
# Global execution guard

# Leave this False when reading or rerunning the notebook for documentation.
# Set to True only when intentionally regenerating the dataset.
RUN_DATA_PIPELINE = False

VSX_CACHE_FOLDER = "VSXCache"
TESS_CACHE_FOLDER = "TESSCache"
VSX_METADATA_FILE = "VSXMetadata.parquet"
AUGMENTED_METADATA_FILE = "TESSAugmented.parquet"

RADIUS_ARCSEC = 5.0
MAX_TIC_CANDIDATES = 5

print("RUN_DATA_PIPELINE =", RUN_DATA_PIPELINE)

## 2.2 Variable-Star Sample and Label Construction

The astrophysical labels in this project come from the **AAVSO International Variable Star Index (VSX)**. VSX provides object names, sky coordinates, periods when available, and detailed variability types.

The raw VSX taxonomy is scientifically rich but too detailed for a first-pass machine-learning dataset. Therefore, related subtypes are grouped into broader astrophysical families. For example, `RRAB`, `RRC`, and `RRD` are grouped into **RRLYR**, while `EA`, `EB`, and `EW` are grouped into **ECLIPSING**.

| Family | Physical interpretation |
|---|---|
| CEPHEID | Radially pulsating stars important for distance-scale work |
| CV | Cataclysmic variables involving accretion onto compact objects |
| DSCT_SXPHE | Short-period pulsators such as Delta Scuti and SX Phoenicis stars |
| ECLIPSING | Binary systems whose brightness changes are caused by eclipses |
| ELLIPSOIDAL_ROT | Rotational, spotted, or tidally distorted systems |
| LONG_PERIOD | Evolved stars with long-timescale variability |
| RRLYR | Horizontal-branch radial pulsators |
| XRAY | High-energy binary or compact-object systems |
| YSO | Young stellar objects with variability from accretion, disks, or spots |

This family-level grouping preserves astrophysical meaning while keeping the sample sizes large enough for statistical comparison.

### Implementation: VSX category loading

The full VSX category-loading code is included below for reproducibility. It is collapsed by default because the scientific logic has already been described above.

In [ ]:
# VSX category loading logic

import sys
import os
import re
import math
import random
import time
import logging
import xml.etree.ElementTree as ET
from io import BytesIO
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from astropy.io.votable import parse_single_table
import lightkurve as lk
import warnings

class VSXCategoryLoader:
    def __init__(self, cacheFolder='VSXCache', refreshCache=False):
        """
        VSXCategoryLoader provides a starter curated dictionary that maps raw
        AAVSO VSX variable-star types into broader normalized families that are
        easier to use for TESS analysis and machine-learning pipelines.

        Core design idea
        ----------------
        The raw VSX type system is astrophysically meaningful but can be too
        granular or inconsistent for a first-pass research pipeline. This class
        adds a curated family layer and provides a family-driven loader:

            input  -> {"ECLIPSING": 100, "RRLYR": 50}
            output -> {
                         "ECLIPSING": [starRecord1, starRecord2, ...],
                         "RRLYR": [starRecord3, starRecord4, ...]
                      }

        Important note
        --------------
        loadCategories() is intentionally family-driven. It is designed to
        retrieve a requested number of variable stars for each normalized family
        by querying the AAVSO VSX service directly.
        """
        self.logger = logging.getLogger("VSXCategoryLoader")
        self.cacheFolder = cacheFolder
        if not os.path.exists(self.cacheFolder):
            os.makedirs(self.cacheFolder)
        self.refreshCache = refreshCache

        self.vsxBaseUrl = "https://www.aavso.org/vsx/index.php"
        self.httpTimeoutSec = 600

        # Keep a reusable session for VSX traffic and automatically recover
        # from transient network/server failures.
        self.httpSession = requests.Session()
        retryPolicy = Retry(
            total=3,
            connect=3,
            read=3,
            status=3,
            backoff_factor=0.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retryPolicy)
        self.httpSession.mount("https://", adapter)
        self.httpSession.mount("http://", adapter)

        # ------------------------------------------------------------
        # Raw VSX type -> normalized family
        # ------------------------------------------------------------
        self.vsxTypeToFamily = {
            # --------------------------------------------------------
            # ECLIPSING BINARIES
            # --------------------------------------------------------
            # Binary-star systems whose brightness variations are caused by
            # eclipses along our line of sight. These are among the most
            # important classes for TESS because they are strongly periodic,
            # often high signal-to-noise, and can resemble exoplanet transits.
            #
            # EA = Algol-type detached binary
            # EB = Beta Lyrae-type semi-detached binary
            # EW = W UMa-type contact binary
            # E  = generic eclipsing binary
            "EA": "ECLIPSING",
            "EB": "ECLIPSING",
            "EW": "ECLIPSING",
            "E": "ECLIPSING",

            # --------------------------------------------------------
            # ROTATIONAL / ELLIPSOIDAL VARIABLES
            # --------------------------------------------------------
            # Variables driven by rotation, star spots, magnetic structure,
            # or tidal distortion rather than eclipses. These are often
            # periodic or quasi-periodic and can look sinusoidal in light
            # curves, making them important comparison cases for pulsators.
            #
            # ELL = ellipsoidal variable
            # RS  = RS CVn active binary
            # BY  = BY Dra spotted dwarf
            # ACV = Alpha2 CVn magnetic variable
            # ROT = generic rotational variable
            "ELL": "ELLIPSOIDAL_ROT",
            "RS": "ELLIPSOIDAL_ROT",
            "BY": "ELLIPSOIDAL_ROT",
            "ACV": "ELLIPSOIDAL_ROT",
            "ROT": "ELLIPSOIDAL_ROT",

            # --------------------------------------------------------
            # RR LYRAE
            # --------------------------------------------------------
            # Old, low-mass radial pulsators on the horizontal branch.
            # These are classic benchmark stars for period-finding and
            # Galactic-structure work, and they often have clean TESS signals.
            #
            # RRAB = fundamental mode
            # RRC  = first overtone
            # RRD  = double-mode
            # RR   = generic RR Lyrae
            "RR": "RRLYR",
            "RRAB": "RRLYR",
            "RRC": "RRLYR",
            "RRD": "RRLYR",

            # --------------------------------------------------------
            # DELTA SCUTI / SX PHOENICIS
            # --------------------------------------------------------
            # Short-period pulsators that can show high-frequency and sometimes
            # multi-mode behavior. These are very useful for testing whether a
            # pipeline can recover shorter timescales reliably.
            #
            # DSCT  = Delta Scuti
            # HADS  = High-Amplitude Delta Scuti
            # SXPHE = SX Phoenicis
            "DSCT": "DSCT_SXPHE",
            "HADS": "DSCT_SXPHE",
            "SXPHE": "DSCT_SXPHE",

            # --------------------------------------------------------
            # CEPHEIDS
            # --------------------------------------------------------
            # Classical and related pulsators with longer periods and strong
            # astrophysical importance because of the period-luminosity relation.
            #
            # DCEP  = Classical Cepheid
            # DCEPS = short-period Cepheid subtype
            # CWA   = Type II Cepheid subtype
            # CWB   = Type II Cepheid subtype
            # ACEP  = Anomalous Cepheid
            "DCEP": "CEPHEID",
            "DCEPS": "CEPHEID",
            "CWA": "CEPHEID",
            "CWB": "CEPHEID",
            "ACEP": "CEPHEID",

            # --------------------------------------------------------
            # LONG-PERIOD VARIABLES
            # --------------------------------------------------------
            # Evolved giant and supergiant stars with long-timescale
            # variability. These can be regular, semi-regular, or irregular,
            # and they are important edge cases because TESS baselines may not
            # fully cover their variability cycles.
            #
            # M   = Mira
            # SRA = semiregular A
            # SRB = semiregular B
            # LB  = slow irregular red variable
            # LPV = generic long-period variable
            "M": "LONG_PERIOD",
            "SRA": "LONG_PERIOD",
            "SRB": "LONG_PERIOD",
            "LB": "LONG_PERIOD",
            "LPV": "LONG_PERIOD",

            # --------------------------------------------------------
            # YOUNG STELLAR OBJECTS
            # --------------------------------------------------------
            # Pre-main-sequence stars whose brightness variations may be caused
            # by accretion, occultation by disk material, magnetic spots, or
            # eruptive behavior. These are often hard ML cases because they may
            # be irregular rather than strictly periodic.
            #
            # TTS  = T Tauri star
            # CTTS = classical T Tauri
            # WTTS = weak-lined T Tauri
            # UXOR = UX Orionis type
            # FUOR = FU Orionis type
            "TTS": "YSO",
            "CTTS": "YSO",
            "WTTS": "YSO",
            "UXOR": "YSO",
            "FUOR": "YSO",

            # --------------------------------------------------------
            # CATACLYSMIC VARIABLES
            # --------------------------------------------------------
            # Interacting binaries with a white dwarf accreting matter.
            # These can show outbursts, eruptions, disk-instability behavior,
            # and more complex light-curve patterns than simple periodic stars.
            #
            # CV = generic cataclysmic variable
            # UG = dwarf nova
            # N  = nova
            "CV": "CV",
            "UG": "CV",
            "N": "CV",

            # --------------------------------------------------------
            # X-RAY BINARIES / HIGH-ENERGY SYSTEMS
            # --------------------------------------------------------
            # Systems involving compact objects such as neutron stars or black
            # holes, where optical variability may reflect accretion, orbital
            # modulation, disk physics, or irradiation. These are rare but
            # scientifically important complex cases.
            #
            # X    = generic X-ray source
            # HMXB = high-mass X-ray binary
            # LMXB = low-mass X-ray binary
            "X": "XRAY",
            "HMXB": "XRAY",
            "LMXB": "XRAY",
        }

        # ------------------------------------------------------------
        # Normalized family metadata
        # ------------------------------------------------------------
        self.familyMetadata = {
            "ECLIPSING": {
                "displayName": "Eclipsing Binaries",
                "physicalMechanism": "Geometric eclipse in a binary system",
                "signalType": "Strongly periodic and often non-sinusoidal",
                "recommendedAlgorithm": "BLS",
                "typicalDifficulty": "easy",
                "tessUseCase": "Excellent for eclipse and transit-like event detection",
                "mlNotes": "Distinct morphology and strong benchmark class",
            },
            "ELLIPSOIDAL_ROT": {
                "displayName": "Rotational / Ellipsoidal Variables",
                "physicalMechanism": "Rotation, spots, or tidal distortion",
                "signalType": "Periodic or quasi-periodic and often smooth",
                "recommendedAlgorithm": "LombScargle",
                "typicalDifficulty": "medium",
                "tessUseCase": "Good for smooth periodic recovery tests",
                "mlNotes": "Can overlap morphologically with pulsators",
            },
            "RRLYR": {
                "displayName": "RR Lyrae Stars",
                "physicalMechanism": "Radial pulsation of horizontal-branch stars",
                "signalType": "Periodic and often asymmetric",
                "recommendedAlgorithm": "LombScargle",
                "typicalDifficulty": "easy",
                "tessUseCase": "Excellent benchmark for period recovery",
                "mlNotes": "Highly recognizable and astrophysically important",
            },
            "DSCT_SXPHE": {
                "displayName": "Delta Scuti / SX Phoenicis",
                "physicalMechanism": "Short-period stellar pulsation",
                "signalType": "Periodic, high-frequency, sometimes multi-mode",
                "recommendedAlgorithm": "LombScargle",
                "typicalDifficulty": "medium",
                "tessUseCase": "Useful for high-frequency recovery studies",
                "mlNotes": "Often benefits from careful frequency-domain features",
            },
            "CEPHEID": {
                "displayName": "Cepheids",
                "physicalMechanism": "Coherent radial pulsation",
                "signalType": "Periodic and relatively regular",
                "recommendedAlgorithm": "LombScargle",
                "typicalDifficulty": "easy",
                "tessUseCase": "Strong classical pulsator benchmark",
                "mlNotes": "Historically important and morphologically structured",
            },
            "LONG_PERIOD": {
                "displayName": "Long-Period Variables",
                "physicalMechanism": "Pulsation and envelope dynamics in evolved stars",
                "signalType": "Long timescale, semi-regular, or irregular",
                "recommendedAlgorithm": "LombScargle",
                "typicalDifficulty": "hard",
                "tessUseCase": "Useful edge case because TESS baseline may be limited",
                "mlNotes": "Long periods may reduce completeness in short baselines",
            },
            "YSO": {
                "displayName": "Young Stellar Objects",
                "physicalMechanism": "Accretion, occultation, disk effects, magnetic activity",
                "signalType": "Irregular or semi-periodic",
                "recommendedAlgorithm": "ML / custom features",
                "typicalDifficulty": "hard",
                "tessUseCase": "Stress-test class for irregular variability",
                "mlNotes": "Important for non-periodic classification experiments",
            },
            "CV": {
                "displayName": "Cataclysmic Variables",
                "physicalMechanism": "Accretion onto a white dwarf",
                "signalType": "Irregular, eruptive, or hybrid periodic",
                "recommendedAlgorithm": "ML / event detection",
                "typicalDifficulty": "hard",
                "tessUseCase": "Useful for accretion-driven outburst behavior",
                "mlNotes": "Often does not fit simple periodic taxonomy",
            },
            "XRAY": {
                "displayName": "X-ray Binaries / High-Energy Systems",
                "physicalMechanism": "Accretion onto compact objects",
                "signalType": "Complex, noisy, or hybrid modulation",
                "recommendedAlgorithm": "ML / custom analysis",
                "typicalDifficulty": "hard",
                "tessUseCase": "Rare but scientifically valuable special class",
                "mlNotes": "Good rare-class challenge set",
            },
            "UNKNOWN": {
                "displayName": "Unknown / Unmapped",
                "physicalMechanism": "Unspecified",
                "signalType": "Unspecified",
                "recommendedAlgorithm": "Review manually",
                "typicalDifficulty": "unknown",
                "tessUseCase": "Requires inspection",
                "mlNotes": "Potential future expansion target",
            },
        }

        # ------------------------------------------------------------
        # Reverse mapping: family -> raw VSX types
        # ------------------------------------------------------------
        self.familyToVsxTypes = defaultdict(list)
        for vsxType, family in self.vsxTypeToFamily.items():
            self.familyToVsxTypes[family].append(vsxType)

    def close(self):
        """
        Explicitly close network resources held by this loader.
        """
        session = getattr(self, "httpSession", None)
        if session is not None:
            session.close()
            self.httpSession = None

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass

    def _httpGetVsxContent(self, params):
        """
        Perform one VSX GET request and always close the response object.

        Returns
        -------
        bytes
            Response body content.
        """
        time.sleep(5)
        vsxResponsePath = os.path.join(self.cacheFolder, f"vsx_response_{params['vtype']}.xml")
        if self.refreshCache or not os.path.exists(vsxResponsePath):
            self.logger.info("Connecting to VSX to download fresh data...")
            with self.httpSession.get(
                self.vsxBaseUrl,
                params=params,
                timeout=self.httpTimeoutSec,
            ) as response:
                response.raise_for_status()
                with open(vsxResponsePath, "wb") as f:
                    f.write(response.content)
                self.logger.info("VSX data downloaded and cached to %s", vsxResponsePath)
                return response.content
        else:
            self.logger.info("Loading VSX data from cache: %s", vsxResponsePath)
            with open(vsxResponsePath, "rb") as f:
                return f.read()

    # ============================================================
    # Internal helpers
    # ============================================================

    def _parseVsxType(self, vsxType):
        """
        Parse a possibly composite VSX type string.

        Example:
            'EA/DSCT|RRAB+ROT' -> ['EA', 'RRAB', 'ROT']

        Notes:
        - '|' often means uncertain alternative
        - '+' often means multiple variability behaviors
        - '/' often indicates compound notation; this starter mapper keeps
          the leading token before '/'
        """
        if not isinstance(vsxType, str):
            return []

        return [
            part.strip().split("/")[0].strip()
            for part in re.split(r"[|+]", vsxType)
            if part.strip()
        ]

    def _mapVsxTypeToFamilies(self, vsxType):
        """
        Map a raw VSX type string to all matching normalized families.

        Returns:
            list[str | None]
            - Each position corresponds to a parsed VSX type
            - None indicates an unknown/unmapped type
        """
        parsedTypes = self._parseVsxType(vsxType)

        return [
            self.vsxTypeToFamily[parsedType]
            if parsedType in self.vsxTypeToFamily
            else None
            for parsedType in parsedTypes
        ]

    def _pickPrimaryFamily(self, families):
        """
        Pick a single primary family from a list of candidate families.

        This is useful for single-label ML datasets. The priority order here
        is a research choice and can be changed later if your project evolves.
        """
        priority = [
            "ECLIPSING",
            "RRLYR",
            "CEPHEID",
            "DSCT_SXPHE",
            "LONG_PERIOD",
            "ELLIPSOIDAL_ROT",
            "YSO",
            "CV",
            "XRAY",
            "UNKNOWN",
        ]

        for family in priority:
            if family in families:
                return family

        return "UNKNOWN"

    def _safeTableValue(self, row, columnName, defaultValue=None):
        """
        Safely extract a value from an Astropy row-like object.
        """
        try:
            return row[columnName]
        except Exception:
            return defaultValue

    def _filterRecordsByVsxTypes(self, records, vsxTypes):
        """
        Return records whose raw VSX type belongs to the requested raw VSX set.

        This method supports both simple raw types like 'EA' and composite raw
        types such as 'EA|DSCT' by parsing the record's vsxType field.
        """
        vsxTypeSet = set(vsxTypes)

        return [
            record
            for record in records
            if any(
                parsedType in vsxTypeSet
                for parsedType in self._parseVsxType(record.get("vsxType", ""))
            )
        ]

    def _loadStarsForFamily(self, family, count):
        """
        Load a requested number of stars for one normalized family directly
        from AAVSO VSX.
        """
        if family not in self.familyToVsxTypes:
            return []

        rawVsxTypes = self.familyToVsxTypes[family]
        if not rawVsxTypes or count <= 0:
            return []

        perTypeTarget = max(1, math.ceil(count / len(rawVsxTypes)))

        combined = []
        for rawVsxType in rawVsxTypes:
            try:
                typeMatches = self._queryVsxByRawType(family, rawVsxType)
            except Exception as exc:
                self.logger.warning("VSX query failed for raw type %s: %s", rawVsxType, exc)
                continue

            if len(typeMatches) > perTypeTarget:
                typeMatches = random.sample(typeMatches, perTypeTarget)

            combined.extend(typeMatches)

            if len(self._deduplicateStars(combined)) >= count:
                break

        combined = self._deduplicateStars(combined)

        if len(combined) > count:
            combined = random.sample(combined, count)

        return combined

    # ============================================================
    # Public metadata helpers
    # ============================================================

    def getSupportedFamilies(self):
        """
        Return the sorted list of supported normalized families.
        """
        return sorted(self.familyMetadata.keys())

    def getFamilyMetadata(self, family):
        """
        Return metadata for one normalized family.
        """
        return self.familyMetadata.get(family, self.familyMetadata["UNKNOWN"])

    def mapVsxTypeToPrimaryFamily(self, vsxType):
        """
        Convenience wrapper for raw VSX type -> single primary family.
        """
        families = self._mapVsxTypeToFamilies(vsxType)
        knownFamilies = [family for family in families if family is not None]
        return self._pickPrimaryFamily(knownFamilies)

    # ============================================================
    # Family-driven category loading
    # ============================================================

    def loadCategories(self, categoryRequests):
        """
        Load variable stars by requested normalized family directly from VSX.

        Parameters
        ----------
        categoryRequests : dict[str, int]
            Example:
                {
                    "ECLIPSING": 100,
                    "RRLYR": 50
                }

        Returns
        -------
        dict[str, list[dict]]
            Dictionary mapping each requested family to a list of fetched stars.
        """
        result = {}

        for family, count in categoryRequests.items():
            if family not in self.familyToVsxTypes:
                self.logger.warning("Unsupported family requested: %s", family)
                result[family] = []
                continue

            result[family] = self._loadStarsForFamily(family=family, count=count)

        return result
    
    def _votableToRowDicts(self, responseContent):
        """
        Parse VSX VOTable response bytes into a list of row dictionaries.
        """
        # VSX responses can omit strict VOTable typing details for string fields,
        # which can cause Astropy to truncate values to one character. Parse the
        # XML table cells directly first to preserve full text values.
        try:
            root = ET.fromstring(responseContent)
            tableElement = root.find(".//{*}TABLE")
            if tableElement is not None:
                fieldElements = tableElement.findall("{*}FIELD")
                columnNames = [
                    fieldElement.get("name") or fieldElement.get("id") or fieldElement.get("ID")
                    for fieldElement in fieldElements
                    if fieldElement.get("name") or fieldElement.get("id") or fieldElement.get("ID")
                ]

                if columnNames:
                    rowElements = tableElement.findall(".//{*}TR")
                    rowDicts = []
                    for rowElement in rowElements:
                        rowValues = [
                            (cell.text or "").strip()
                            for cell in rowElement.findall("{*}TD")
                        ]
                        rowDicts.append({
                            columnName: (rowValues[index] if index < len(rowValues) else "")
                            for index, columnName in enumerate(columnNames)
                        })

                    if rowDicts:
                        return rowDicts
        except Exception as exc:
            self.logger.debug("XML VOTable parsing path failed, falling back to Astropy: %s", exc)

        # Fallback: keep the Astropy path for compatibility.
        table = parse_single_table(BytesIO(responseContent)).to_table(use_names_over_ids=True)
        columnNames = list(table.colnames)

        return [
            {
                columnName: row[columnName].item() if hasattr(row[columnName], "item") else row[columnName]
                for columnName in columnNames
            }
            for row in table
        ]

    def _queryVsxByRawType(self, family, rawVsxType):
        """
        Query AAVSO VSX for one raw variability type and return normalized records.
        """
        params = {
            "view": "query.votable",
            "vtype": rawVsxType,
        }

        responseContent = self._httpGetVsxContent(params)
        rowDicts = self._votableToRowDicts(responseContent)

        return [
            self._normalizeVsxRow(rowDict, rawVsxType, family)
            for rowDict in rowDicts
        ]
    
    def _deduplicateStars(self, stars):
        """
        Deduplicate combined results across multiple raw VSX types.
        """
        seenKeys = set()
        deduped = []

        for star in stars:
            key = (
                star.get("VSXId"),
                star.get("VSXName"),
                star.get("raDeg"),
                star.get("decDeg"),
            )
            if key in seenKeys:
                continue
            seenKeys.add(key)
            deduped.append(star)

        return deduped
    
    def _pickFirstExistingKey(self, rowDict, candidateKeys):
        lowerKeyMap = {key.lower(): key for key in rowDict.keys()}

        for candidate in candidateKeys:
            actualKey = lowerKeyMap.get(candidate.lower())
            if actualKey is not None:
                return rowDict.get(actualKey)

        return None
    
    def _safeFloat(self, value):
        try:
            if value is None or value == "":
                return None
            return float(value)
        except Exception:
            return None
    
    def _parseCoordsJ2000(self, coordText):
        """
        Parse VSX 'Coords(J2000)' field formatted like:
            '11.44133333,41.84172222'
        into:
            (raDeg, decDeg)
        """
        if isinstance(coordText, bytes):
            coordText = coordText.decode("utf-8", errors="ignore")

        if not isinstance(coordText, str):
            return None, None

        coordText = coordText.strip()
        if not coordText:
            return None, None

        # Primary expected format from VSX is: "ra,dec".
        if "," in coordText:
            parts = [part.strip() for part in coordText.split(",")]
        else:
            # Fallback for whitespace-delimited variants.
            parts = coordText.split()

        if len(parts) != 2:
            return None, None

        try:
            raDeg = float(parts[0])
            decDeg = float(parts[1])
            return raDeg, decDeg
        except ValueError:
            return None, None
    
    def _normalizeVsxRow(self, rowDict, requestedRawVsxType, family):
        """
        Normalize one VSX VOTable row into the internal record shape.
        """
        coordText = self._pickFirstExistingKey(
            rowDict,
            ["Coords(J2000)", "radec2000"]
        )
        raDeg, decDeg = self._parseCoordsJ2000(coordText)

        VSXId = self._pickFirstExistingKey(rowDict, ["AUID", "Name"])
        VSXName = self._pickFirstExistingKey(rowDict, ["Name"])
        vsxType = self._pickFirstExistingKey(rowDict, ["VarType"])
        period = self._safeFloat(self._pickFirstExistingKey(rowDict, ["Period"]))

        return {
            "VSXId": VSXId if VSXId not in [None, ""] else VSXName,
            "VSXName": VSXName,
            "VSXType": vsxType if vsxType not in [None, ""] else requestedRawVsxType,
            "period": period,
            "family": family,
            "raDeg": raDeg,
            "decDeg": decDeg,
            "rawRow": rowDict,
        }
    
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    loader = VSXCategoryLoader()

    categoryRequests = {
        "ECLIPSING": 20,
        "RRLYR": 20,
        "DSCT_SXPHE": 20,
    }

    try:
        supportedFamilies = set(loader.getSupportedFamilies())
        unsupportedFamilies = [
            family for family in categoryRequests.keys()
            if family not in supportedFamilies
        ]
        if unsupportedFamilies:
            loader.logger.warning(
                "Requested unsupported families: %s",
                unsupportedFamilies,
            )

        categoryDictionary = loader.loadCategories(categoryRequests=categoryRequests)

        totalLoaded = sum(len(stars) for stars in categoryDictionary.values())
        loader.logger.info("Loaded %d stars across %d families", totalLoaded, len(categoryDictionary))

        for family, stars in categoryDictionary.items():
            requestedCount = categoryRequests.get(family)
            loader.logger.info(
                "Family %s: requested=%s loaded=%d",
                family,
                requestedCount,
                len(stars),
            )

            if stars:
                sample = stars[0]
                loader.logger.info(
                    "Sample %s star: VSXName=%s VSXType=%s RA=%.6f Dec=%.6f",
                    family,
                    sample.get("VSXName"),
                    sample.get("VSXType"),
                    sample.get("raDeg") if sample.get("raDeg") is not None else float("nan"),
                    sample.get("decDeg") if sample.get("decDeg") is not None else float("nan"),
                )
    finally:
        loader.close()

## 2.3 Cross-Matching VSX Objects to the TESS Input Catalog

VSX identifies stars by sky position and variable-star name, while TESS light-curve products are usually accessed through TIC identifiers. Therefore, each VSX object is associated with likely TESS Input Catalog entries.

The adopted cross-match strategy is:

- use the VSX J2000 coordinates,
- search the TIC within a **5.0 arcsecond** radius,
- retain up to **five** TIC candidates,
- sort candidates by angular separation,
- preserve all candidate distances for later reliability analysis.

The decision to retain up to five TIC candidates is important. If only the nearest TIC object were kept, some stars would lose usable TESS coverage simply because the nearest object lacks a SPOC or QLP product. Retaining multiple nearby candidates allows the pipeline to recover available TESS products while still making the match-confidence information transparent.

The 5.0 arcsecond radius is narrow enough to avoid broad ambiguous sky associations, but flexible enough to account for catalog-position uncertainties, proper-motion effects, and crowded-field complications.

### Implementation: VSX-to-TIC cross-match

The full cross-match implementation is included below and collapsed by default.

In [ ]:
# VSX-to-TIC cross-match logic

from astropy.coordinates import SkyCoord
from astropy import units as u
from astroquery.mast import Catalogs, Observations
import logging

class VSX2TESSConverter:
    """
    VSX2TESSConverter provides utilities to convert AAVSO VSX variable-star
    records into a format suitable for crossmatching with TESS Input Catalog
    (TIC) entries and downstream TESS analysis pipelines.

    This class is intended to take a VSX record (as returned by VSXCategoryLoader)
    and produce a record enriched with TIC information, including TIC ID, TESS
    magnitude, and angular separation from the VSX coordinates.
    """

    def __init__(self):
        self.logger = logging.getLogger("VSX2TESSConverter")

    def crossmatchToTic(self, starRecord, radiusArcsec=5.0, maxMatches=5):
        """
        Crossmatch AAVSO VSX variable-star record to the TESS Input Catalog using RA/Dec.

        The returned record includes a `ticCandidates` list containing up to
        `maxMatches` candidates sorted by angular separation from the VSX target.
        Each candidate has: ticId, ticRaDeg, ticDecDeg, ticTmag, ticDistanceArcmin.

        Returns
        -------
        dict or None
            A copy of the input record with TIC fields appended,
            or None if no TIC match is found.
        """
        raDeg = starRecord.get("raDeg")
        decDeg = starRecord.get("decDeg")

        if raDeg is None or decDeg is None:
            return None

        coord = SkyCoord(ra=raDeg * u.deg, dec=decDeg * u.deg, frame="icrs")

        try:
            ticTable = Catalogs.query_region(
                coord,
                radius=radiusArcsec * u.arcsec,
                catalog="TIC",
            )
        except Exception as exc:
            self.logger.warning(
                "TIC crossmatch failed for %s: %s",
                starRecord.get("VSXName"),
                exc,
            )
            return None

        if ticTable is None or len(ticTable) == 0:
            return None

        # TIC query results do not always provide a query-center "distance"
        # column, so compute angular separation explicitly from RA/Dec.
        hasRa = "ra" in ticTable.colnames
        hasDec = "dec" in ticTable.colnames

        ticCandidates = []
        if hasRa and hasDec:
            for row in ticTable:
                try:
                    ticRaDeg = float(row["ra"])
                    ticDecDeg = float(row["dec"])
                    ticTmagRaw = row["Tmag"]
                    try:
                        ticTmag = float(ticTmagRaw)
                    except Exception:
                        ticTmag = None

                    
                    ticCoord = SkyCoord(
                        ra=float(ticRaDeg) * u.deg,
                        dec=float(ticDecDeg) * u.deg,
                        frame="icrs"
                    )
                    separationArcsec = float(coord.separation(ticCoord).arcsec)
                except Exception:
                    continue

                ticCandidates.append(
                    {
                        "ticId": str(row["ID"]),
                        "ticRaDeg": ticRaDeg,
                        "ticDecDeg": ticDecDeg,
                        "ticTmag": ticTmag,
                        "ticDistanceArcmin": separationArcsec / 60.0,
                    }
                )

        if not ticCandidates:
            return None

        maxMatches = max(1, int(maxMatches))
        #be sure to sort by ticDistanceArcmin
        ticCandidates.sort(key=lambda candidate: candidate["ticDistanceArcmin"])
        topCandidates = ticCandidates[:maxMatches]

        matchedRecord = dict(starRecord)
        matchedRecord["ticCandidates"] = topCandidates
        matchedRecord['VSXId'] = starRecord.get('VSXId')
        matchedRecord['VSXName'] = starRecord.get('VSXName')
        matchedRecord['VSXType'] = starRecord.get('VSXType')
        matchedRecord['family'] = starRecord.get('family')
        matchedRecord['period'] = starRecord.get('period')

        return matchedRecord

    def crossmatchCategoryDictionaryToTic(self, categoryDictionary, radiusArcsec=5.0):
        """
        Crossmatch all stars in a family->stars dictionary to TIC.
        """
        result = {}

        for family, stars in categoryDictionary.items():
            self.logger.info("Crossmatching family '%s' with %d stars to TIC", family, len(stars))
            result[family] = [
                matchedRecord
                for star in stars
                if (matchedRecord := self.crossmatchToTic(star, radiusArcsec=radiusArcsec)) is not None
            ]

        self.findBestMatches(result)

        return result
    
    def findBestMatches(self, categoryDictionary, batchSize=200):
        """
        For each star, inspect TIC candidates in distance order and choose the first
        one with an available TESS light curve, preferring SPOC over QLP.
        Adds a 'bestMatch' field with {ticId, author}, or None if neither author exists.
        """
        def _normalize_tic(value):
            if value is None:
                return None
            digits = "".join(ch for ch in str(value) if ch.isdigit())
            if not digits:
                return None
            return str(int(digits))

        def _chunked(items, chunkSize):
            for idx in range(0, len(items), chunkSize):
                yield items[idx : idx + chunkSize]

        uniqueTicIds = sorted(
            {
                _normalize_tic(candidate.get("ticId"))
                for stars in categoryDictionary.values()
                for star in stars
                for candidate in star.get("ticCandidates", [])
                if _normalize_tic(candidate.get("ticId")) is not None
            }
        )

        ticAvailability = {
            ticId: {"SPOC": False, "QLP": False}
            for ticId in uniqueTicIds
        }

        batchSize = max(1, int(batchSize))
        for ticBatch in _chunked(uniqueTicIds, batchSize):
            try:
                observations = Observations.query_criteria(
                    project=["TESS"],
                    provenance_name=["SPOC", "QLP"],
                    dataproduct_type=["timeseries", "cube"],
                    target_name=[ticId.zfill(9) for ticId in ticBatch],
                )
            except Exception as exc:
                self.logger.warning(
                    "Availability query failed for TIC batch of size %d: %s",
                    len(ticBatch),
                    exc,
                )
                continue

            if observations is None:
                continue

            for row in observations:
                ticId = _normalize_tic(row.get("target_name"))
                author = str(row.get("provenance_name", "")).strip().upper()
                if ticId not in ticAvailability or author not in {"SPOC", "QLP"}:
                    continue
                ticAvailability[ticId][author] = True

        for family, stars in categoryDictionary.items():
            for star in stars:
                ticCandidates = star.get("ticCandidates", [])
                star["bestMatch"] = None

                for candidate in ticCandidates:
                    ticId = _normalize_tic(candidate.get("ticId"))
                    if ticId is None:
                        continue

                    availability = ticAvailability.get(ticId, {"SPOC": False, "QLP": False})

                    if availability.get("SPOC"):
                        star["bestMatch"] = {
                            "ticId": ticId,
                            "author": "SPOC",
                        }
                        break

                    if availability.get("QLP"):
                        star["bestMatch"] = {
                            "ticId": ticId,
                            "author": "QLP",
                        }
                        break
    

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    converter = VSX2TESSConverter()

    def _check_crossmatch(starRecord, expectedTicId, label):
        tessRecord = converter.crossmatchToTic(starRecord)
        if tessRecord is None:
            converter.logger.error("%s crossmatch failed: no TIC candidates returned", label)
            return

        ticCandidates = tessRecord.get("ticCandidates", [])
        if not ticCandidates:
            converter.logger.error("%s crossmatch failed: empty ticCandidates", label)
            return

        bestTicId = str(ticCandidates[0].get("ticId"))
        if bestTicId != str(expectedTicId):
            converter.logger.error(
                "%s crossmatch failed: expected TIC ID %s, got %s",
                label,
                expectedTicId,
                bestTicId,
            )
        else:
            converter.logger.info("%s crossmatch succeeded: TIC ID %s", label, bestTicId)

    _check_crossmatch(
        {"raDeg": 291.36629, "decDeg": 42.78436, "VSXName": ""},
        "159717514",
        "RR Lyr",
    )
    _check_crossmatch(
        {"raDeg": 214.15242, "decDeg": 42.35992, "VSXName": ""},
        "168709463",
        "TV Boo",
    )
    _check_crossmatch(
        {"raDeg": 319.84242, "decDeg": 38.23747, "VSXName": ""},
        "373202340",
        "V1334 Cyg",
    )
    _check_crossmatch(
        {"raDeg": 135.78246, "decDeg": 44.58558, "VSXName": ""},
        "29172806",
        "TT Lyn",
    )
    _check_crossmatch(
        {"raDeg": 321.00100, "decDeg": 18.27883, "VSXName": ""},
        "279587090",
        "AU Peg",
    )

## 2.4 Light-Curve Acquisition and Photometric Provenance

After TIC candidates are selected, the pipeline attempts to obtain a usable TESS light curve. The provenance priority is:

1. **SPOC**
2. **QLP**
3. **TESSCut**

This priority creates a controlled fallback system. SPOC and QLP are used when standard light-curve products exist. If neither is available, the pipeline extracts photometry from TESS Full-Frame Image cutouts using the Lightkurve aperture-photometry workflow.

This fallback step is scientifically important because many VSX objects do not have standard SPOC or QLP light curves. Without TESSCut, the dataset would be much smaller and more strongly biased toward stars already covered by standard products.

For every successful acquisition, the pipeline records selected TIC ID, provenance category, light-curve file paths, normalization metadata, quality-mask information, and sector information when available.

Both raw and standardized light curves are retained. Standardization places light curves on a comparable numerical scale for feature extraction, but it does **not** perform detrending.

### Implementation: TESS light-curve acquisition

The full downloader implementation is included below and collapsed by default.

In [ ]:
# TESS light-curve download and SPOC -> QLP -> TESSCut fallback logic

import pandas as pd
import lightkurve as lk
import logging
import os
import glob
import re
import sys
import time
import shutil
import io
import numpy as np
import threading
import warnings
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from contextlib import redirect_stderr, redirect_stdout
from concurrent.futures import ThreadPoolExecutor, as_completed


class _QualityMaskLogFilter(logging.Filter):
    """
    Drops quality-mask log records emitted by Lightkurve below a given
    ignored-cadence percentage threshold, and updates downloader quality.
    """
    _PATTERN = re.compile(
        r"([0-9]+(?:\.[0-9]+)?)%.*cadences will be ignored due to the quality mask",
        re.IGNORECASE,
    )

    def __init__(self, thresholdPercent=20.0, downloader=None):
        super().__init__()
        self.thresholdPercent = thresholdPercent
        self.downloader = downloader

    def filter(self, record):
        message = record.getMessage()
        match = self._PATTERN.search(message)
        if match is not None:
            percentage = float(match.group(1))
            if self.downloader is not None:
                self.downloader._updateStarQuality(percentage)
            return percentage > self.thresholdPercent
        return True


_qualityMaskLogFilter = _QualityMaskLogFilter(thresholdPercent=20.0, downloader=None)
for _lkLoggerName in (
    "lightkurve",
    "lightkurve.io",
    "lightkurve.lightcurve",
    "lightkurve.utils",
    "lightkurve.search",
    "lightkurve.targetpixelfile",
):
    logging.getLogger(_lkLoggerName).addFilter(_qualityMaskLogFilter)

class TessDataDownloader:
    """
    A class to download TESS light curve data organized by stellar categories.
    """
    
    def __init__(self, tessCacheFolder='TESSCache'):
        """Initialize the TessDataDownloader."""
        self.logger = logging.getLogger("TessDataDownloader")
        self.tessCacheFolder = tessCacheFolder
        self.normalizationStdFloor = 1e-8
        self.lowSnrThreshold = 3.0
        if not os.path.exists(self.tessCacheFolder):
            os.makedirs(self.tessCacheFolder)

        self.ignoredCadenceWarningThresholdPercent = 20.0
        self._threadState = threading.local()
        self._workerDownloadsRoot = os.path.join(self.tessCacheFolder, "_worker_downloads")
        
        global _qualityMaskLogFilter
        _qualityMaskLogFilter.downloader = self

    def preprocessLightCurve(
        self,
        lightCurve,
        doRemoveNans=True,
        doNormalize=True,
        doFlatten=False,
        flattenWindowLength=401,
        doRemoveOutliers=False,
        sigma=5.0,
        doBin=False,
        timeBinSize=0.01,
    ):
        """
        Standard preprocessing helper for TESS light curves.

        Keep this configurable because different science cases benefit from
        different preprocessing choices.
        """
        if lightCurve is None:
            return None

        processed = lightCurve
        normalizationMetadata = None

        try:
            if doRemoveNans:
                processed = processed.remove_nans()

            if doNormalize:
                processed, normalizationMetadata = self._standardizeLightCurve(processed)
                if processed is None:
                    return None

            if doFlatten:
                processed = processed.flatten(window_length=flattenWindowLength)

            if doRemoveOutliers:
                processed = processed.remove_outliers(sigma=sigma)

            if doBin:
                processed = processed.bin(time_bin_size=timeBinSize)

            if doRemoveNans:
                processed = processed.remove_nans()

        except Exception as exc:
            self.logger.warning("preprocessLightCurve failed: %s", exc)
            return None

        return processed

    def _resolveMetadataPath(self, tessMetadataParquet):
        if os.path.isabs(tessMetadataParquet):
            return tessMetadataParquet

        cachePath = os.path.join(self.tessCacheFolder, tessMetadataParquet)
        if os.path.exists(cachePath):
            return cachePath

        return tessMetadataParquet

    def _cleanupTransientDownloads(self, downloadRoot=None):
        rootPath = downloadRoot or self.tessCacheFolder
        transientPaths = [
            os.path.join(rootPath, "mastDownload"),
            os.path.join(rootPath, "tesscut"),
        ]

        for transientPath in transientPaths:
            if not os.path.exists(transientPath):
                continue

            try:
                shutil.rmtree(transientPath)
            except Exception as exc:
                self.logger.warning(
                    "Failed to remove transient download directory %s: %s",
                    transientPath,
                    exc,
                )

    def _resetStarQuality(self):
        self._threadState.currentStarQuality = "missing"

    def _getCurrentStarQuality(self):
        return getattr(self._threadState, "currentStarQuality", "missing")

    def _normalizeTicId(self, value):
        if value is None:
            return None

        try:
            if pd.isna(value):
                return None
        except TypeError:
            pass

        digits = "".join(ch for ch in str(value) if ch.isdigit())
        if not digits:
            return None

        return str(int(digits))

    def _sanitizeFilenameComponent(self, value):
        """Sanitize a string for use in filenames by replacing spaces and special chars with underscores."""
        if value is None:
            return None
        # Replace spaces and other problematic characters with underscores
        sanitized = str(value).replace(" ", "_").replace(",", "_").replace(":", "_")
        return sanitized

    def _sortedTicCandidates(self, ticCandidates):
        if ticCandidates is None:
            return []

        if isinstance(ticCandidates, float) and pd.isna(ticCandidates):
            return []

        if hasattr(ticCandidates, "tolist"):
            ticCandidates = ticCandidates.tolist()

        if isinstance(ticCandidates, dict):
            ticCandidates = [ticCandidates]

        normalizedCandidates = []
        for candidate in ticCandidates or []:
            if candidate is None:
                continue
            normalizedCandidates.append(dict(candidate))

        normalizedCandidates.sort(
            key=lambda candidate: float(candidate.get("ticDistanceArcmin", float("inf")))
        )
        return normalizedCandidates

    def _logDuplicateTicMappings(self, df):
        """Log stars that map to the same top TIC candidate."""
        starsByTopTic = {}
        duplicateWarningCount = 0

        for _, row in df.iterrows():
            ticCandidates = self._sortedTicCandidates(row.get("ticCandidates"))
            topTic = self._normalizeTicId((ticCandidates[0] if ticCandidates else {}).get("ticId"))
            if topTic is None:
                continue

            candidateTics = [
                self._normalizeTicId(candidate.get("ticId"))
                for candidate in ticCandidates
            ]
            candidateTics = [tic for tic in candidateTics if tic is not None]

            starsByTopTic.setdefault(topTic, []).append(
                {
                    "family": row.get("family"),
                    "VSXType": row.get("VSXType"),
                    "VSXId": row.get("VSXId"),
                    "tics": candidateTics,
                }
            )

        for topTic, stars in starsByTopTic.items():
            if len(stars) < 2:
                continue
            for star in stars:
                self.logger.error(
                    "Duplicate TIC mapping detected: TIC=%s family=%s vsxtype=%s vsxid=%s tics=%s",
                    topTic,
                    star.get("family"),
                    star.get("VSXType"),
                    star.get("VSXId"),
                    star.get("tics"),
                )
                duplicateWarningCount += 1

        return duplicateWarningCount

    def _standardizeLightCurve(self, lightCurve):
        if lightCurve is None:
            return None, None

        cleanedLightCurve = lightCurve.remove_nans()
        if len(cleanedLightCurve) == 0:
            return None, {
                "normalizationApplied": False,
                "fluxMedian": None,
                "fluxStd": None,
                "fluxSnr": None,
                "medianUnstable": True,
                "lowSNR": True,
                "lowQualityLightCurve": True,
                "lowQualityReason": "No valid cadences after removing NaNs",
                "validCadenceCount": 0,
            }

        fluxArray = np.asarray(getattr(cleanedLightCurve.flux, "value", cleanedLightCurve.flux), dtype=float)
        validMask = np.isfinite(fluxArray)

        fluxMask = getattr(cleanedLightCurve.flux, "mask", None)
        if fluxMask is not None:
            fluxMaskArray = np.asarray(fluxMask, dtype=bool)
            if fluxMaskArray.shape == ():
                validMask &= (not bool(fluxMaskArray))
            else:
                validMask &= np.logical_not(fluxMaskArray)

        if hasattr(cleanedLightCurve.time, "value"):
            timeValues = np.asarray(cleanedLightCurve.time.value, dtype=float)
            validMask &= np.isfinite(timeValues)

        validFlux = fluxArray[validMask]
        if validFlux.size == 0:
            return None, {
                "normalizationApplied": False,
                "fluxMedian": None,
                "fluxStd": None,
                "fluxSnr": None,
                "medianUnstable": True,
                "lowSNR": True,
                "lowQualityLightCurve": True,
                "lowQualityReason": "No valid cadences after masking invalid flux values",
                "validCadenceCount": 0,
            }

        medianFlux = float(np.nanmedian(validFlux))
        stdFlux = float(np.nanstd(validFlux))
        medianUnstable = (not np.isfinite(medianFlux)) or abs(medianFlux) < self.normalizationStdFloor
        lowStd = (not np.isfinite(stdFlux)) or stdFlux < self.normalizationStdFloor
        fluxSnr = None if lowStd else float(abs(medianFlux) / stdFlux)
        lowSNR = fluxSnr is None or fluxSnr < self.lowSnrThreshold

        normalizationMetadata = {
            "normalizationApplied": False,
            "fluxMedian": medianFlux,
            "fluxStd": stdFlux,
            "fluxSnr": fluxSnr,
            "medianUnstable": bool(medianUnstable),
            "lowSNR": bool(lowSNR),
            "lowQualityLightCurve": bool(medianUnstable or lowSNR or lowStd),
            "lowQualityReason": None,
            "validCadenceCount": int(validFlux.size),
        }

        if lowStd:
            normalizationMetadata["lowQualityReason"] = "Flux standard deviation is too small for stable normalization"
            return cleanedLightCurve, normalizationMetadata

        standardizedFlux = np.full_like(fluxArray, np.nan, dtype=float)
        standardizedFlux[validMask] = (fluxArray[validMask] - medianFlux) / stdFlux

        standardizedLightCurve = cleanedLightCurve.copy()
        standardizedLightCurve.flux = standardizedFlux * u.dimensionless_unscaled

        if hasattr(standardizedLightCurve, "flux_err") and standardizedLightCurve.flux_err is not None:
            fluxErrArray = np.asarray(
                getattr(standardizedLightCurve.flux_err, "value", standardizedLightCurve.flux_err),
                dtype=float,
            )
            standardizedFluxErr = np.full_like(fluxErrArray, np.nan, dtype=float)
            finiteFluxErr = np.isfinite(fluxErrArray)
            standardizedFluxErr[finiteFluxErr] = fluxErrArray[finiteFluxErr] / stdFlux
            standardizedLightCurve.flux_err = standardizedFluxErr * u.dimensionless_unscaled

        normalizationMetadata["normalizationApplied"] = True
        return standardizedLightCurve.remove_nans(), normalizationMetadata

    def _storeLightCurve(self, lightCurve, outputFile, ticId, authorLabel):
        try:
            lightCurve.to_fits(path=outputFile, overwrite=True)
        except (AttributeError, ValueError) as exc:
            self.logger.warning(
                "to_fits failed for TIC %s author %s, using fallback: %s",
                ticId,
                authorLabel,
                exc,
            )
            try:
                timeCol = fits.Column(name="TIME", format="D", array=lightCurve.time.jd)
                fluxCol = fits.Column(name="FLUX", format="E", array=np.asarray(lightCurve.flux.value, dtype=float))
                cols = fits.ColDefs([timeCol, fluxCol])
                hdu = fits.BinTableHDU.from_columns(cols)
                hdu.writeto(outputFile, overwrite=True)
            except Exception as fallbackExc:
                self.logger.error(
                    "Fallback FITS write also failed for TIC %s author %s: %s",
                    ticId,
                    authorLabel,
                    fallbackExc,
                )
                return False

        return True

    def _categorizeQualityMaskPercentage(self, percentage):
        """Categorize data quality based on quality_bitmask cadence percentage."""
        if percentage < 10:
            return "clean"
        elif percentage < 25:
            return "acceptable"
        elif percentage < 40:
            return "caution"
        else:
            return "poor"

    def _updateStarQuality(self, percentage):
        """Update current star quality to worst quality seen so far."""
        newQuality = self._categorizeQualityMaskPercentage(percentage)
        qualityOrder = {"clean": 0, "acceptable": 1, "caution": 2, "poor": 3}
        currentQuality = self._getCurrentStarQuality()
        if qualityOrder.get(newQuality, 0) > qualityOrder.get(currentQuality, 0):
            self._threadState.currentStarQuality = newQuality

    def _finalizeStarQuality(self, lightCurveAvailable):
        """Return final quality after considering whether any light curve was available."""
        if not lightCurveAvailable:
            return "missing"

        if self._getCurrentStarQuality() == "missing":
            return "clean"

        return self._getCurrentStarQuality()

    def _runWithFilteredWarnings(self, func, *args, ticId=None, warningContext=None, **kwargs):
        qualityWarningPattern = re.compile(
            r"([0-9]+(?:\.[0-9]+)?)%\s*\([^)]*\)\s*of the cadences will be ignored due to the quality mask",
            re.IGNORECASE,
        )
        negativeMedianPattern = re.compile(
            r"negative median flux",
            re.IGNORECASE,
        )
        zeroCenteredPattern = re.compile(
            r"zero-centered.*normalize\(\)",
            re.IGNORECASE,
        )
        boolInversionDeprecationPattern = re.compile(
            r"bitwise inversion\s+['`~]+\s*on bool\s+is deprecated",
            re.IGNORECASE,
        )

        with warnings.catch_warnings(record=True) as caughtWarnings:
            warnings.simplefilter("always")
            result = func(*args, **kwargs)

        for caughtWarning in caughtWarnings:
            warningMessage = str(caughtWarning.message)
            qualityMatch = qualityWarningPattern.search(warningMessage)

            if qualityMatch is not None:
                ignoredPercent = float(qualityMatch.group(1))
                self._updateStarQuality(ignoredPercent)
                if ignoredPercent > self.ignoredCadenceWarningThresholdPercent:
                    self.logger.warning(
                        "High quality-mask rejection for TIC %s%s: %s",
                        ticId if ticId is not None else "unknown",
                        f" during {warningContext}" if warningContext else "",
                        warningMessage,
                    )
                continue

            if negativeMedianPattern.search(warningMessage) is not None:
                continue

            if zeroCenteredPattern.search(warningMessage) is not None:
                continue

            if boolInversionDeprecationPattern.search(warningMessage) is not None:
                continue

            self.logger.warning(
                "Warning for TIC %s%s: %s",
                ticId if ticId is not None else "unknown",
                f" during {warningContext}" if warningContext else "",
                warningMessage,
            )

        return result

    def _lightCurveFromSearchResult(self, searchResult, ticId=None, downloadDir=None):
        if searchResult is None or len(searchResult) == 0:
            return None

        resolvedDownloadDir = downloadDir or self.tessCacheFolder

        try:
            downloaded = self._runWithFilteredWarnings(
                searchResult.download_all,
                download_dir=resolvedDownloadDir,
                ticId=ticId,
                warningContext="search_lightcurve download_all",
            )
        except Exception:
            downloaded = self._runWithFilteredWarnings(
                searchResult.download,
                download_dir=resolvedDownloadDir,
                ticId=ticId,
                warningContext="search_lightcurve download",
            )

        if downloaded is None:
            return None

        if isinstance(downloaded, lk.LightCurveCollection):
            if len(downloaded) == 0:
                return None
            try:
                return downloaded.stitch()
            except Exception as exc:
                self.logger.warning("Failed to stitch light-curve collection: %s", exc)
                return downloaded[0]

        return downloaded

    def _runQuietLightkurveSearch(self, searchFn, *args, **kwargs):
        loggerNames = [
            "lightkurve.search",
            "astroquery",
            "astroquery.mast",
        ]
        loggerState = []

        for loggerName in loggerNames:
            packageLogger = logging.getLogger(loggerName)
            loggerState.append((packageLogger, packageLogger.level, packageLogger.disabled))
            packageLogger.setLevel(logging.CRITICAL + 1)

        try:
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                return searchFn(*args, **kwargs)
        finally:
            for packageLogger, level, disabled in loggerState:
                packageLogger.setLevel(level)
                packageLogger.disabled = disabled

    def _downloadCatalogLightCurve(self, ticId, author, downloadDir=None, vsxId=None):
        try:
            searchResult = self._runQuietLightkurveSearch(
                lk.search_lightcurve,
                f"TIC {ticId}",
                mission="TESS",
                author=author,
            )
        except Exception as exc:
            self.logger.warning(
                "search_lightcurve failed for TIC %s author %s: %s",
                ticId,
                author,
                exc,
            )
            return None

        if searchResult is None or len(searchResult) == 0:
            return None

        lightCurve = self._lightCurveFromSearchResult(
            searchResult,
            ticId=ticId,
            downloadDir=downloadDir,
        )
        if lightCurve is None:
            return None

        # Save raw light curve
        sanitizedVsxId = self._sanitizeFilenameComponent(vsxId)
        vsxIdStr = f"VSX_{sanitizedVsxId}_" if sanitizedVsxId else ""
        rawOutputFile = os.path.join(self.tessCacheFolder, f"{vsxIdStr}TIC_{ticId}_{author}_raw.fits")
        if not self._storeLightCurve(lightCurve, rawOutputFile, ticId, f"{author}_raw"):
            return None

        # Standardize light curve
        standardizedLightCurve, normalizationMetadata = self._standardizeLightCurve(lightCurve)
        if standardizedLightCurve is None:
            return None

        # Save standardized light curve
        standardizedOutputFile = os.path.join(self.tessCacheFolder, f"{vsxIdStr}TIC_{ticId}_{author}_standardized.fits")
        if not self._storeLightCurve(standardizedLightCurve, standardizedOutputFile, ticId, f"{author}_standardized"):
            return None

        sectors = []
        if hasattr(searchResult, "table") and "sequence_number" in searchResult.table.colnames:
            sectors = [
                int(sector)
                for sector in searchResult.table["sequence_number"]
                if sector is not None and str(sector) != "--"
            ]

        return {
            "bestMatch": {"ticId": ticId, "author": author},
            "provenance": author,
            "lightCurvePath": standardizedOutputFile,  # Standardized by default
            "rawLightCurvePath": rawOutputFile,
            "lightCurveAvailable": True,
            "extractionMetadata": {
                "downloadMethod": "search_lightcurve",
                "author": author,
                "productCount": int(len(searchResult)),
                "sectors": sectors,
                "fluxNormalization": normalizationMetadata,
            },
            "normalizationApplied": normalizationMetadata["normalizationApplied"],
            "fluxMedian": normalizationMetadata["fluxMedian"],
            "fluxStd": normalizationMetadata["fluxStd"],
            "fluxSnr": normalizationMetadata["fluxSnr"],
            "medianUnstable": normalizationMetadata["medianUnstable"],
            "lowSNR": normalizationMetadata["lowSNR"],
            "lowQualityLightCurve": normalizationMetadata["lowQualityLightCurve"],
            "lowQualityReason": normalizationMetadata["lowQualityReason"],
        }

    def _extractTessCutLightCurve(self, starRecord, ticCandidates, cutoutSize, downloadDir=None):
        selectedCandidate = {}
        selectedCandidateRank = None
        sourceRaDeg = None
        sourceDecDeg = None

        sortedCandidates = self._sortedTicCandidates(ticCandidates)

        # Use the first TIC candidate that has both coordinates populated.
        for rank, candidate in enumerate(sortedCandidates, start=1):
            candidateRaDeg = candidate.get("ticRaDeg")
            candidateDecDeg = candidate.get("ticDecDeg")
            if candidateRaDeg is None or candidateDecDeg is None:
                continue
            selectedCandidate = candidate
            selectedCandidateRank = rank
            sourceRaDeg = candidateRaDeg
            sourceDecDeg = candidateDecDeg
            break

        # Fall back to original VSX coordinates only when no TIC candidate has coordinates.
        if sourceRaDeg is None or sourceDecDeg is None:
            sourceRaDeg = starRecord.get("raDeg")
            sourceDecDeg = starRecord.get("decDeg")

        # Preserve prior behavior for TIC-based metadata if no candidate had usable coordinates.
        if not selectedCandidate:
            selectedCandidate = (sortedCandidates[0] if sortedCandidates else {})

        if selectedCandidateRank is not None:
            self.logger.info(
                "TESSCut fallback will use TIC candidate rank %d TIC=%s distance=%s arcmin for VSX=%s",
                selectedCandidateRank,
                selectedCandidate.get("ticId"),
                selectedCandidate.get("ticDistanceArcmin"),
                starRecord.get("VSXId"),
            )
        else:
            self.logger.info(
                "TESSCut fallback has no TIC candidate with valid coordinates; using original VSX coords for VSX=%s",
                starRecord.get("VSXId"),
            )

        if sourceRaDeg is None or sourceDecDeg is None:
            return None

        resolvedDownloadDir = downloadDir or self.tessCacheFolder

        try:
            coord = SkyCoord(ra=float(sourceRaDeg) * u.deg, dec=float(sourceDecDeg) * u.deg, frame="icrs")
        except Exception as exc:
            self.logger.warning(
                "Invalid sky coordinates for %s: %s",
                starRecord.get("VSXName", starRecord.get("VSXId", "unknown")),
                exc,
            )
            return None

        try:
            searchResult = self._runQuietLightkurveSearch(lk.search_tesscut, coord)
        except Exception as exc:
            self.logger.warning(
                "search_tesscut failed for %s: %s",
                starRecord.get("VSXName", starRecord.get("VSXId", "unknown")),
                exc,
            )
            return None

        if searchResult is None or len(searchResult) == 0:
            return None

        try:
            tpfCollection = self._runWithFilteredWarnings(
                searchResult.download_all,
                cutout_size=cutoutSize,
                download_dir=resolvedDownloadDir,
                ticId=self._normalizeTicId(selectedCandidate.get("ticId")),
                warningContext="search_tesscut download_all",
            )
        except Exception as exc:
            self.logger.warning(
                "TESSCut download failed for %s: %s",
                starRecord.get("VSXName", starRecord.get("VSXId", "unknown")),
                exc,
            )
            return None

        if tpfCollection is None or len(tpfCollection) == 0:
            return None

        extractedCurves = []
        aperturePixelCounts = []
        sectors = []

        for tpf in tpfCollection:
            try:
                apertureMask = tpf.create_threshold_mask(threshold=3, reference_pixel="center")
                if apertureMask is None or not np.any(apertureMask):
                    apertureMask = np.zeros(tpf.flux[0].shape, dtype=bool)
                    apertureMask[apertureMask.shape[0] // 2, apertureMask.shape[1] // 2] = True

                lightCurve = self._runWithFilteredWarnings(
                    tpf.to_lightcurve,
                    aperture_mask=apertureMask,
                    ticId=self._normalizeTicId(selectedCandidate.get("ticId")),
                    warningContext="TESSCut to_lightcurve",
                ).remove_nans()
                if len(lightCurve) == 0:
                    continue

                extractedCurves.append(lightCurve)
                aperturePixelCounts.append(int(np.sum(apertureMask)))

                sector = getattr(tpf, "sector", None)
                if sector is not None:
                    sectors.append(int(sector))
            except Exception as exc:
                self.logger.warning(
                    "Aperture photometry failed for %s in one TESSCut sector: %s",
                    starRecord.get("VSXName", starRecord.get("VSXId", "unknown")),
                    exc,
                )
            finally:
                try:
                    if hasattr(tpf, "close") and callable(tpf.close):
                        tpf.close()
                    elif hasattr(tpf, "hdu") and hasattr(tpf.hdu, "close"):
                        tpf.hdu.close()
                except Exception as closeExc:
                    self.logger.debug("Failed to close TESSCut handle cleanly: %s", closeExc)

        if not extractedCurves:
            return None

        if len(extractedCurves) == 1:
            stitched = extractedCurves[0]
        else:
            try:
                stitched = lk.LightCurveCollection(extractedCurves).stitch()
            except Exception as exc:
                self.logger.warning("Failed to stitch TESSCut light curves: %s", exc)
                stitched = extractedCurves[0]

        stitched = stitched.remove_nans()
        if len(stitched) == 0:
            return None

        # Save raw light curve
        chosenTicId = self._normalizeTicId(selectedCandidate.get("ticId")) or "NA"
        vsxId = starRecord.get("VSXId", "NA")
        sanitizedVsxId = self._sanitizeFilenameComponent(vsxId)
        rawOutputFile = os.path.join(self.tessCacheFolder, f"VSX_{sanitizedVsxId}_TIC_{chosenTicId}_TESSCut_raw.fits")
        if not self._storeLightCurve(stitched, rawOutputFile, chosenTicId, "TESSCut_raw"):
            return None

        # Standardize light curve
        standardizedLightCurve, normalizationMetadata = self._standardizeLightCurve(stitched)
        if standardizedLightCurve is None:
            return None

        # Save standardized light curve
        standardizedOutputFile = os.path.join(self.tessCacheFolder, f"VSX_{sanitizedVsxId}_TIC_{chosenTicId}_TESSCut_standardized.fits")
        if not self._storeLightCurve(standardizedLightCurve, standardizedOutputFile, chosenTicId, "TESSCut_standardized"):
            return None

        return {
            "bestMatch": {"ticId": chosenTicId, "author": "TESSCut"},
            "provenance": "TESSCut",
            "lightCurvePath": standardizedOutputFile,  # Standardized by default
            "rawLightCurvePath": rawOutputFile,
            "lightCurveAvailable": True,
            "extractionMetadata": {
                "downloadMethod": "search_tesscut",
                "sourceRaDeg": float(sourceRaDeg),
                "sourceDecDeg": float(sourceDecDeg),
                "cutoutSize": list(cutoutSize),
                "sectorCount": len(extractedCurves),
                "sectors": sectors,
                "aperturePixelCounts": aperturePixelCounts,
                "selectedCandidateTicId": chosenTicId,
                "selectedCandidateDistanceArcmin": selectedCandidate.get("ticDistanceArcmin"),
                "extractionMethod": "threshold_mask_photometry",
                "fluxNormalization": normalizationMetadata,
            },
            "normalizationApplied": normalizationMetadata["normalizationApplied"],
            "fluxMedian": normalizationMetadata["fluxMedian"],
            "fluxStd": normalizationMetadata["fluxStd"],
            "fluxSnr": normalizationMetadata["fluxSnr"],
            "medianUnstable": normalizationMetadata["medianUnstable"],
            "lowSNR": normalizationMetadata["lowSNR"],
            "lowQualityLightCurve": normalizationMetadata["lowQualityLightCurve"],
            "lowQualityReason": normalizationMetadata["lowQualityReason"],
        }

    def _processSingleStar(self, idx, rowData, count, totalCount, cutoutSize, cleanupDownloads):
        starName = rowData.get("VSXName", rowData.get("VSXId", f"row-{idx}"))
        family = rowData.get("family")
        self.logger.info(
            "Processing star %d/%d: %s (family=%s)",
            count,
            totalCount,
            starName,
            family,
        )

        self._resetStarQuality()
        os.makedirs(self._workerDownloadsRoot, exist_ok=True)
        taskDownloadDir = os.path.join(
            self._workerDownloadsRoot,
            f"star_{idx}_{threading.get_ident()}_{int(time.time() * 1000)}",
        )
        os.makedirs(taskDownloadDir, exist_ok=True)

        updates = {
            "bestMatch": None,
            "provenance": None,
            "lightCurvePath": None,
            "rawLightCurvePath": None,
            "extractionMetadata": None,
            "lightCurveAvailable": False,
            "noLightCurveReason": None,
            "normalizationApplied": None,
            "fluxMedian": None,
            "fluxStd": None,
            "fluxSnr": None,
            "medianUnstable": None,
            "lowSNR": None,
            "lowQualityLightCurve": None,
            "lowQualityReason": None,
            "quality": "missing",
        }

        # Check if files already exist
        vsxId = rowData.get("VSXId")
        if vsxId:
            sanitized = self._sanitizeFilenameComponent(vsxId)
            std_pattern = os.path.join(self.tessCacheFolder, f"VSX_{sanitized}_*standardized.fits")
            std_files = glob.glob(std_pattern)
            if std_files:
                std_path = std_files[0]
                raw_path = std_path.replace('_standardized.fits', '_raw.fits')
                if os.path.exists(raw_path):
                    updates["lightCurvePath"] = std_path
                    updates["rawLightCurvePath"] = raw_path
                    updates["lightCurveAvailable"] = True
                    updates["provenance"] = "existing"
                    updates["quality"] = "existing"
                    self.logger.info("Skipping %s: files already exist", starName)
                    return idx, updates

        ticCandidates = self._sortedTicCandidates(rowData.get("ticCandidates"))
        preferredTicId = self._normalizeTicId(
            (ticCandidates[0] if ticCandidates else {}).get("ticId")
        )
        matchedResult = None

        try:
            def _starTask():
                nonlocal matchedResult

                for candidate in ticCandidates:
                    ticId = self._normalizeTicId(candidate.get("ticId"))
                    if ticId is None:
                        continue

                    for author in ("SPOC", "QLP"):
                        matchedResult = self._downloadCatalogLightCurve(
                            ticId,
                            author,
                            downloadDir=taskDownloadDir,
                            vsxId=rowData.get("VSXId"),
                        )
                        if matchedResult is None:
                            continue

                        matchedResult["bestMatch"]["ticDistanceArcmin"] = candidate.get("ticDistanceArcmin")
                        matchedResult["bestMatch"]["ticRaDeg"] = candidate.get("ticRaDeg")
                        matchedResult["bestMatch"]["ticDecDeg"] = candidate.get("ticDecDeg")
                        matchedResult["bestMatch"]["ticTmag"] = candidate.get("ticTmag")
                        matchedResult["extractionMetadata"]["selectedCandidate"] = dict(candidate)
                        return

                matchedResult = self._extractTessCutLightCurve(
                    rowData,
                    ticCandidates,
                    cutoutSize,
                    downloadDir=taskDownloadDir,
                )

            self._runWithFilteredWarnings(
                _starTask,
                ticId=preferredTicId,
                warningContext="star processing",
            )

            if matchedResult is None:
                self.logger.error(
                    "No usable TESS light curve found for %s after SPOC, QLP, and TESSCut attempts",
                    starName,
                )
                updates["noLightCurveReason"] = (
                    "No SPOC/QLP light curve and no usable TESSCut extraction"
                )
                updates["quality"] = self._finalizeStarQuality(False)
                return idx, updates

            updates.update(
                {
                    "bestMatch": matchedResult["bestMatch"],
                    "provenance": matchedResult["provenance"],
                    "lightCurvePath": matchedResult["lightCurvePath"],
                    "rawLightCurvePath": matchedResult.get("rawLightCurvePath"),
                    "extractionMetadata": matchedResult["extractionMetadata"],
                    "lightCurveAvailable": matchedResult["lightCurveAvailable"],
                    "normalizationApplied": matchedResult.get("normalizationApplied"),
                    "fluxMedian": matchedResult.get("fluxMedian"),
                    "fluxStd": matchedResult.get("fluxStd"),
                    "fluxSnr": matchedResult.get("fluxSnr"),
                    "medianUnstable": matchedResult.get("medianUnstable"),
                    "lowSNR": matchedResult.get("lowSNR"),
                    "lowQualityLightCurve": matchedResult.get("lowQualityLightCurve"),
                    "lowQualityReason": matchedResult.get("lowQualityReason"),
                    "quality": self._finalizeStarQuality(
                        matchedResult.get("lightCurveAvailable", False)
                    ),
                }
            )
            return idx, updates
        finally:
            if cleanupDownloads:
                self._cleanupTransientDownloads(taskDownloadDir)

            try:
                shutil.rmtree(taskDownloadDir)
            except FileNotFoundError:
                pass
            except Exception as exc:
                self.logger.warning(
                    "Failed to remove worker download directory %s: %s",
                    taskDownloadDir,
                    exc,
                )

    def downloadTessLightCurves(
        self,
        tessMetadataParquet,
        augmentedMetadataFile="TESSAugmented.parquet",
        cutoutSize=(15, 15),
        cleanupDownloads=True,
        lowSnrThreshold=None,
        maxWorkers=None,
    ):
        """
        Download or extract TESS light curves for each star in cached metadata.

        The method first tries TIC candidates in angular-separation order using
        SPOC then QLP products. If neither exists, it falls back to TESSCut using
        the original VSX coordinates and writes augmented metadata to parquet.
        """
        originalLowSnrThreshold = self.lowSnrThreshold
        if lowSnrThreshold is not None:
            self.lowSnrThreshold = float(lowSnrThreshold)

        metadataPath = self._resolveMetadataPath(tessMetadataParquet)
        df = pd.read_parquet(metadataPath)

        requiredColumns = {"family", "raDeg", "decDeg", "ticCandidates"}
        missingColumns = requiredColumns - set(df.columns)
        if missingColumns:
            raise ValueError(
                f"Missing required columns in {metadataPath}: {sorted(missingColumns)}"
            )

        # Check if augmented metadata exists to resume from previous run
        augmentedPath = os.path.join(self.tessCacheFolder, augmentedMetadataFile)
        if os.path.exists(augmentedPath):
            self.logger.info("Loading existing augmented metadata from %s to resume processing", augmentedPath)
            df = pd.read_parquet(augmentedPath)
        else:
            # Initialize columns if starting fresh
            for column in [
                "bestMatch",
                "provenance",
                "lightCurvePath",
                "rawLightCurvePath",
                "extractionMetadata",
                "lightCurveAvailable",
                "noLightCurveReason",
                "normalizationApplied",
                "fluxMedian",
                "fluxStd",
                "fluxSnr",
                "medianUnstable",
                "lowSNR",
                "lowQualityLightCurve",
                "lowQualityReason",
                "quality",
            ]:
                if column not in df.columns:
                    df[column] = [None] * len(df)

        duplicateWarningCount = self._logDuplicateTicMappings(df)

        # Only process stars that don't have both raw and standardized files
        orderedIndices = df.sort_values(["family", "VSXName"], na_position="last").index.tolist()
        def has_both_files(row):
            lc_path = row.get("lightCurvePath")
            raw_path = row.get("rawLightCurvePath")
            return lc_path and raw_path and os.path.exists(lc_path) and os.path.exists(raw_path)
        orderedIndices = [idx for idx in orderedIndices if not has_both_files(df.loc[idx])]

        if not orderedIndices:
            self.logger.info("All stars already have light curves available. Skipping download.")
            successCount = int(df["lightCurveAvailable"].eq(True).sum())
            totalCount = int(len(df))
            failedCount = totalCount - successCount
            self.logger.info(
                "Download summary: total=%d succeeded=%d failed=%d duplicateTicWarnings=%d",
                totalCount,
                successCount,
                failedCount,
                duplicateWarningCount,
            )
            return df

        workerCount = maxWorkers
        if workerCount is None:
            workerCount = min(4, max(1, len(orderedIndices)))

        try:
            if workerCount == 1:
                for count, idx in enumerate(orderedIndices, start=1):
                    try:
                        _, updates = self._processSingleStar(
                            idx,
                            df.loc[idx].to_dict(),
                            count,
                            len(orderedIndices),
                            cutoutSize,
                            cleanupDownloads,
                        )
                    except Exception as exc:
                        row = df.loc[idx]
                        starName = row.get("VSXName", row.get("VSXId", f"row-{idx}"))
                        self.logger.exception("Star task failed for %s: %s", starName, exc)
                        updates = {
                            "bestMatch": None,
                            "provenance": None,
                            "lightCurvePath": None,
                            "rawLightCurvePath": None,
                            "extractionMetadata": None,
                            "lightCurveAvailable": False,
                            "noLightCurveReason": f"Worker task failed: {exc}",
                            "normalizationApplied": None,
                            "fluxMedian": None,
                            "fluxStd": None,
                            "fluxSnr": None,
                            "medianUnstable": None,
                            "lowSNR": None,
                            "lowQualityLightCurve": None,
                            "lowQualityReason": None,
                            "quality": "missing",
                        }
                    for column, value in updates.items():
                        df.at[idx, column] = value
                    # Save progress after each star
                    df.to_parquet(augmentedPath, index=False)
            else:
                with ThreadPoolExecutor(max_workers=workerCount) as executor:
                    futureToIdx = {
                        executor.submit(
                            self._processSingleStar,
                            idx,
                            df.loc[idx].to_dict(),
                            count,
                            len(orderedIndices),
                            cutoutSize,
                            cleanupDownloads,
                        ): idx
                        for count, idx in enumerate(orderedIndices, start=1)
                    }

                    for future in as_completed(futureToIdx):
                        idx = futureToIdx[future]
                        try:
                            _, updates = future.result()
                        except Exception as exc:
                            row = df.loc[idx]
                            starName = row.get("VSXName", row.get("VSXId", f"row-{idx}"))
                            self.logger.exception("Star task failed for %s: %s", starName, exc)
                            updates = {
                                "bestMatch": None,
                                "provenance": None,
                                "lightCurvePath": None,
                                "rawLightCurvePath": None,
                                "extractionMetadata": None,
                                "lightCurveAvailable": False,
                                "noLightCurveReason": f"Worker task failed: {exc}",
                                "normalizationApplied": None,
                                "fluxMedian": None,
                                "fluxStd": None,
                                "fluxSnr": None,
                                "medianUnstable": None,
                                "lowSNR": None,
                                "lowQualityLightCurve": None,
                                "lowQualityReason": None,
                                "quality": "missing",
                            }

                        for column, value in updates.items():
                            df.at[idx, column] = value
                        # Save progress after each star
                        df.to_parquet(augmentedPath, index=False)
        finally:
            self.lowSnrThreshold = originalLowSnrThreshold
            if cleanupDownloads:
                self._cleanupTransientDownloads()
            try:
                shutil.rmtree(self._workerDownloadsRoot)
            except FileNotFoundError:
                pass
            except Exception as exc:
                self.logger.warning(
                    "Failed to remove worker download root %s: %s",
                    self._workerDownloadsRoot,
                    exc,
                )

        augmentedPath = os.path.join(self.tessCacheFolder, augmentedMetadataFile)
        df.to_parquet(augmentedPath, index=False)
        successCount = int(df["lightCurveAvailable"].eq(True).sum())
        totalCount = int(len(df))
        failedCount = totalCount - successCount
        self.logger.info(
            "Download summary: total=%d succeeded=%d failed=%d duplicateTicWarnings=%d",
            totalCount,
            successCount,
            failedCount,
            duplicateWarningCount,
        )
        self.logger.info("Saved augmented metadata to %s", augmentedPath)
        return df

## 2.5 Pipeline Driver and Safe Reproducibility

The dataset can be regenerated from the notebook, but this operation is intentionally disabled by default. It requires VSX queries, MAST/TESS queries, light-curve downloads, and file writes. Therefore, the notebook uses the global flag:

```python
RUN_DATA_PIPELINE = False
```

When this flag is `False`, the notebook defines code and displays results but does not regenerate the dataset.

### Implementation: pipeline driver

The driver class is included below and collapsed by default.

In [ ]:
# Pipeline driver class

from VSXCategoryLoader import VSXCategoryLoader
from VSX2TESSConverter import VSX2TESSConverter
from TessDataDownloader import TessDataDownloader
from collections import defaultdict
import pandas as pd
import logging
import sys
import os
import argparse
import numpy as np

class ResearchPipeline:
    """
    ResearchPipeline orchestrates the loading of VSX variable-star categories,
    crossmatching them to the TESS Input Catalog, and writing the resulting
    TIC IDs to a CSV file for downstream analysis.
    """
    def __init__(self, vsxLoader, tessConverter, nFamilies = None, nInstances=1000, vsxCacheFolder='VSXCache', tessCacheFolder='TESSCache'):
        self.loader = vsxLoader
        self.converter = tessConverter
        self.vsxCacheFolder = vsxCacheFolder
        self.tessCacheFolder = tessCacheFolder
        if nFamilies is not None:
            self.varFamilies = self.loader.getSupportedFamilies()[:nFamilies]
        else:
            self.varFamilies = self.loader.getSupportedFamilies()
        self.nInstances = nInstances
        self.ticFile = None
        self.tessMatchedDict = None
        self.logger = logging.getLogger()

    def combineVSXTESS(self):
        #This is a time consuming operation, as it involves crossmatching potentially
        #thousands of VSX variable stars against the TESS Input Catalog
        #This function is supposed to called only once to generate the CSV of TIC IDs
        #for downstream analysis, since the crossmatching step is expensive

        varReq = {fam: self.nInstances for fam in self.varFamilies}

        self.logger.info("Loading categories from VSX")
        categoryDictionary = self.loader.loadCategories(varReq)

        self.logger.info("Crossmatching categories to TIC")
        self.tessMatchedDict = self.converter.crossmatchCategoryDictionaryToTic(categoryDictionary)
        self.tessMatchedDict = {
            family: stars for family, stars in self.tessMatchedDict.items() if stars
        }
        return self.tessMatchedDict
    
    def cacheMetadata(self, tessMarchedDict, metadataFile='VSXMetadata.parquet'):
        df = []
        for family, stars in tessMarchedDict.items():
            for star in stars:
                record = dict(star)
                df.append(record)
        df = pd.DataFrame(df)
        df.to_parquet(self.tessCacheFolder + os.path.sep + metadataFile, index=False)

    def loadTicFile(self, ticFile="tess_matched_stars.csv"):
        """
        Loads a previously generated CSV of TIC IDs into memory for downstream analysis.
        """
        self.ticFile = self.tessCacheFolder + os.path.sep + ticFile
        self.tessMatchedDict = {}
        with open(self.ticFile, "r") as f:
            for line in f:
                family, *ticIds = line.strip().split(",")
                self.tessMatchedDict[family] = [{"ticId": ticId} for ticId in ticIds]
        
        return self.tessMatchedDict
    
    def loadCachedMetadata(self, metadataFile="VSXMetadata.parquet"):
        self.metadataFile = self.tessCacheFolder + os.path.sep + metadataFile
        if os.path.exists(self.metadataFile):
            df = pd.read_parquet(self.metadataFile)
            self.tessMatchedDict = defaultdict(list)
            for _, row in df.iterrows():
                family = row.get("family")
                if family is not None:
                    starRecord = row.to_dict()
                    self.tessMatchedDict[family].append(starRecord)
            return self.tessMatchedDict
        else:
            self.logger.warning(f"Metadata file {self.metadataFile} does not exist.")
            return None
    
    def loadCandidates(self):
        """
        Load candidates from cache if available, otherwise run the full pipeline
        from scratch: loads VSX categories, crossmatches them to the TESS Input Catalog,
        and caches the metadata for downstream analysis.
        """
        metadata_file_path = os.path.join(self.tessCacheFolder, "VSXMetadata.parquet")
        if os.path.exists(metadata_file_path):
            return self.loadCachedMetadata()
        else:
            tessData = self.combineVSXTESS()
            self.cacheMetadata(tessData, metadataFile="VSXMetadata.parquet")
            return tessData

    def countBestMatchesByFamily(self, tessMetadataParquet):
        """
        Count stars with a valid bestMatch for each family from cached parquet metadata.

        Returns:
            tuple[dict[str, int], int]:
                - per_family_count: number of stars with bestMatch found in each family
                - total_best_matches: total number of stars with bestMatch found
        """
        df = pd.read_parquet(tessMetadataParquet)

        required_columns = {"family", "bestMatch"}
        missing_columns = required_columns - set(df.columns)
        if missing_columns:
            raise ValueError(
                f"Missing required columns in {tessMetadataParquet}: {sorted(missing_columns)}"
            )

        # Treat null/NaN and literal string 'None' as no match.
        has_match = df["bestMatch"].notna() & (df["bestMatch"].astype(str) != "None")

        per_family_count = (
            df.loc[has_match]
            .groupby("family")
            .size()
            .astype(int)
            .to_dict()
        )
        total_best_matches = int(has_match.sum())

        return per_family_count, total_best_matches

In [ ]:
# Optional full data generation cell.
# This cell does nothing unless RUN_DATA_PIPELINE is changed to True.

if RUN_DATA_PIPELINE:
    import logging

    logging.basicConfig(level=logging.INFO)

    vsx_loader = VSXCategoryLoader(
        cacheFolder=VSX_CACHE_FOLDER,
        refreshCache=False,
    )
    tess_converter = VSX2TESSConverter()

    pipeline = ResearchPipeline(
        vsxLoader=vsx_loader,
        tessConverter=tess_converter,
        nFamilies=None,
        nInstances=1000,
        vsxCacheFolder=VSX_CACHE_FOLDER,
        tessCacheFolder=TESS_CACHE_FOLDER,
    )

    tess_metadata = pipeline.loadCandidates()
    pipeline.cacheMetadata(tess_metadata, metadataFile=VSX_METADATA_FILE)

    tess_downloader = TessDataDownloader(tessCacheFolder=TESS_CACHE_FOLDER)
    augmented = tess_downloader.downloadTessLightCurves(
        tessMetadataParquet=VSX_METADATA_FILE,
        augmentedMetadataFile=AUGMENTED_METADATA_FILE,
        cutoutSize=(15, 15),
        cleanupDownloads=True,
        maxWorkers=4,
    )

    print("Dataset generation complete.")
    print("Rows:", len(augmented))
else:
    print("Data generation skipped because RUN_DATA_PIPELINE is False.")

## 2.6 FITS Integrity Checks and Quality Control

Because the dataset depends on thousands of downloaded or extracted FITS files, automated sanity checks are essential. A corrupted or truncated FITS file can create artificial features, cause feature-extraction failures, or bias model performance.

The FITS sanity scan checks whether each file opens successfully, contains a recognizable time-flux table, has enough finite time and flux rows, has positive time coverage, has nonzero flux scatter, and is not suspiciously small or metadata-only.

The quality-control results are written back into the augmented metadata table so that problematic light curves can be flagged or excluded later.

This stage is still part of dataset construction. It is not detrending. It only checks whether the light-curve files are structurally usable.

### Implementation: FITS sanity scan

The full FITS scanning and metadata QC code is included below and collapsed by default.

In [ ]:
# FITS sanity-scan and QC logic

#!/usr/bin/env python3
"""
fits_sanity_scan.py

Recursively scan a folder for FITS files and flag suspicious files using checks
similar to the TESS/TESSCut notebook logic.

Usage:
    python fits_sanity_scan.py /path/to/folder
    python fits_sanity_scan.py /path/to/folder --csv report.csv
    python fits_sanity_scan.py /path/to/folder --min-size-kb 100 --min-finite-rows 100
"""

from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from astropy.io import fits


TIME_COL_CANDIDATES = ["TIME", "BJD", "TIMECORR"]
FLUX_COL_CANDIDATES = ["FLUX", "SAP_FLUX", "PDCSAP_FLUX", "KSPSAP_FLUX", "RAW_FLUX"]
FITS_SUFFIXES = {".fits", ".fit", ".fts", ".fits.gz", ".fit.gz", ".fts.gz"}


@dataclass
class ScanResult:
    path: str
    size_bytes: int
    verdict: str
    suspicious: bool
    has_time_flux_table: bool
    has_image_hdu: bool
    hdu_count: int
    table_hdu_index: Optional[int]
    time_col: Optional[str]
    flux_col: Optional[str]
    total_rows: Optional[int]
    finite_rows: Optional[int]
    finite_fraction: Optional[float]
    time_span: Optional[float]
    flux_std: Optional[float]
    issues: str


def looks_like_fits(path: Path) -> bool:
    name = path.name.lower()
    return any(name.endswith(sfx) for sfx in FITS_SUFFIXES)


def find_time_flux_pair(hdul) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    for i, hdu in enumerate(hdul):
        data = hdu.data
        names = getattr(data, "names", None)
        if names is None:
            continue
        names_upper = {n.upper(): n for n in names}
        time_col = next((names_upper[t] for t in TIME_COL_CANDIDATES if t in names_upper), None)
        flux_col = next((names_upper[f] for f in FLUX_COL_CANDIDATES if f in names_upper), None)
        if time_col and flux_col:
            return i, time_col, flux_col
    return None, None, None


def has_image_like_hdu(hdul) -> bool:
    for hdu in hdul:
        data = hdu.data
        if isinstance(data, np.ndarray) and data.ndim >= 2 and data.size > 0:
            return True
    return False


def safe_float(value) -> Optional[float]:
    try:
        val = float(value)
        if np.isnan(val):
            return None
        return val
    except Exception:
        return None


def analyze_table(hdul, hdu_idx: int, time_col: str, flux_col: str):
    table = hdul[hdu_idx].data
    total_rows = len(table)

    try:
        time = np.asarray(table[time_col], dtype=float)
        flux = np.asarray(table[flux_col], dtype=float)
    except Exception:
        return total_rows, None, None, None

    finite_mask = np.isfinite(time) & np.isfinite(flux)
    finite_rows = int(finite_mask.sum())
    finite_fraction = finite_rows / total_rows if total_rows else None

    if finite_rows == 0:
        return total_rows, finite_rows, finite_fraction, None, None

    time_f = time[finite_mask]
    flux_f = flux[finite_mask]

    time_span = safe_float(np.nanmax(time_f) - np.nanmin(time_f))
    flux_std = safe_float(np.nanstd(flux_f))
    return total_rows, finite_rows, finite_fraction, time_span, flux_std


def classify_and_flag(
    path: Path,
    min_size_kb: int,
    min_finite_rows: int,
    min_finite_fraction: float,
) -> ScanResult:
    issues: List[str] = []
    size_bytes = path.stat().st_size

    if size_bytes < min_size_kb * 1024:
        issues.append(f"small_file<{min_size_kb}KB")

    try:
        with fits.open(path) as hdul:
            hdu_count = len(hdul)
            image_hdu = has_image_like_hdu(hdul)
            hdu_idx, time_col, flux_col = find_time_flux_pair(hdul)

            has_time_flux = hdu_idx is not None

            if has_time_flux:
                total_rows, finite_rows, finite_fraction, time_span, flux_std = analyze_table(
                    hdul, hdu_idx, time_col, flux_col
                )
            else:
                total_rows = finite_rows = None
                finite_fraction = time_span = flux_std = None

            if has_time_flux:
                verdict = "LIKELY_FINAL_LIGHT_CURVE_FITS"
            elif image_hdu:
                verdict = "LIKELY_IMAGE_OR_TESSCUT_PRODUCT"
            else:
                verdict = "UNCLEAR_OR_METADATA_ONLY"

            if not has_time_flux:
                issues.append("no_recognizable_time_flux_table")

            if has_time_flux and total_rows is not None and total_rows == 0:
                issues.append("empty_time_flux_table")

            if has_time_flux and finite_rows is not None and finite_rows < min_finite_rows:
                issues.append(f"too_few_finite_rows<{min_finite_rows}")

            if has_time_flux and finite_fraction is not None and finite_fraction < min_finite_fraction:
                issues.append(f"low_finite_fraction<{min_finite_fraction:.2f}")

            if has_time_flux and time_span is not None and time_span <= 0:
                issues.append("non_positive_time_span")

            if has_time_flux and flux_std is not None and flux_std == 0:
                issues.append("zero_flux_scatter")

            suspicious = len(issues) > 0

            return ScanResult(
                path=str(path),
                size_bytes=size_bytes,
                verdict=verdict,
                suspicious=suspicious,
                has_time_flux_table=has_time_flux,
                has_image_hdu=image_hdu,
                hdu_count=hdu_count,
                table_hdu_index=hdu_idx,
                time_col=time_col,
                flux_col=flux_col,
                total_rows=total_rows,
                finite_rows=finite_rows,
                finite_fraction=finite_fraction,
                time_span=time_span,
                flux_std=flux_std,
                issues=";".join(issues),
            )

    except Exception as e:
        return ScanResult(
            path=str(path),
            size_bytes=size_bytes,
            verdict="FAILED_TO_OPEN",
            suspicious=True,
            has_time_flux_table=False,
            has_image_hdu=False,
            hdu_count=0,
            table_hdu_index=None,
            time_col=None,
            flux_col=None,
            total_rows=None,
            finite_rows=None,
            finite_fraction=None,
            time_span=None,
            flux_std=None,
            issues=f"open_error:{type(e).__name__}:{e}",
        )


def scan_folder(folder: Path, min_size_kb: int, min_finite_rows: int, min_finite_fraction: float) -> List[ScanResult]:
    results: List[ScanResult] = []
    for path in folder.rglob("*"):
        if path.is_file() and looks_like_fits(path):
            results.append(classify_and_flag(path, min_size_kb, min_finite_rows, min_finite_fraction))
    return results


def write_csv(results: List[ScanResult], csv_path: Path) -> None:
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(results[0]).keys()) if results else list(ScanResult.__annotations__.keys()))
        writer.writeheader()
        for r in results:
            writer.writerow(asdict(r))


def print_summary(results: List[ScanResult]) -> None:
    total = len(results)
    suspicious = sum(r.suspicious for r in results)
    failed = sum(r.verdict == "FAILED_TO_OPEN" for r in results)
    lightcurve = sum(r.verdict == "LIKELY_FINAL_LIGHT_CURVE_FITS" for r in results)
    image_like = sum(r.verdict == "LIKELY_IMAGE_OR_TESSCUT_PRODUCT" for r in results)
    unclear = sum(r.verdict == "UNCLEAR_OR_METADATA_ONLY" for r in results)

    print(f"Total FITS files scanned: {total}")
    print(f"Suspicious files: {suspicious}")
    print(f"Failed to open: {failed}")
    print(f"Likely final light curve FITS: {lightcurve}")
    print(f"Likely image/TESSCut products: {image_like}")
    print(f"Unclear/metadata only: {unclear}")

    if suspicious:
        print("\nSuspicious files:")
        for r in results:
            if r.suspicious:
                print(f"- {r.path}")
                print(f"  verdict={r.verdict}, size={r.size_bytes} bytes, issues={r.issues}")


def determine_qc_status(scan_result: ScanResult) -> Tuple[str, str]:
    """
    Determine QC status based on scan results.
    
    Returns:
        Tuple[str, str]: (status, details) where status is "pass", "warning", or "fail"
    """
    issues = []
    status = "pass"
    
    # Hard fail conditions
    if scan_result.finite_rows is not None and scan_result.finite_rows < 100:
        issues.append("too_few_finite_rows")
        status = "fail"
    
    if scan_result.finite_fraction is not None and scan_result.finite_fraction < 0.50:
        issues.append("low_finite_fraction")
        status = "fail"
    
    # Soft warning conditions (only if not already failed)
    if status != "fail":
        warning_flags = []
        
        if scan_result.finite_fraction is not None and 0.50 <= scan_result.finite_fraction < 0.75:
            warning_flags.append("marginal_finite_fraction")
        
        if scan_result.finite_rows is not None and 100 <= scan_result.finite_rows < 300:
            warning_flags.append("marginal_row_count")
        
        if scan_result.size_bytes < 50 * 1024:  # 50 KB
            warning_flags.append("small_file")
        
        # Warn if any soft conditions are met
        if len(warning_flags) > 0:
            status = "warning"
            issues.extend(warning_flags)
    
    # Check for hard failure indicators
    if scan_result.verdict == "FAILED_TO_OPEN":
        status = "fail"
        issues.append("failed_to_open")
    
    if scan_result.verdict == "UNCLEAR_OR_METADATA_ONLY":
        if not scan_result.has_time_flux_table:
            status = "fail"
            issues.append("no_time_flux_table")
    
    details = ";".join(issues) if issues else "all_checks_passed"
    return status, details


def process_augmented_parquet(folder: Path) -> None:
    """
    Process TESSAugmented.parquet: scan associated FITS files and add QC columns.
    """
    parquet_path = folder / "TESSAugmented.parquet"
    output_path = folder / "TESSAugmented_QC.parquet"
    
    if not parquet_path.exists():
        print(f"Parquet file not found: {parquet_path}")
        return
    
    print(f"Loading {parquet_path}...")
    df = pd.read_parquet(parquet_path)
    
    # Add QC columns
    df["fitsQcStatus"] = "unknown"
    df["fitsQcReason"] = ""
    
    print(f"Processing {len(df)} rows for FITS QC...")
    
    for idx, row in df.iterrows():
        if idx % 100 == 0:
            print(f"  Processing row {idx}/{len(df)}...")
        
        # Get light curve paths
        lc_path = row.get("lightCurvePath")
        raw_path = row.get("rawLightCurvePath")
        
        # Skip if no paths or both missing
        if pd.isna(lc_path) or not lc_path:
            df.at[idx, "fitsQcStatus"] = "unknown"
            df.at[idx, "fitsQcReason"] = "no_light_curve_path"
            continue
        
        # Check if files exist
        lc_file = Path(lc_path)
        if not lc_file.exists():
            df.at[idx, "fitsQcStatus"] = "fail"
            df.at[idx, "fitsQcReason"] = "light_curve_file_missing"
            continue
        
        # Scan the light curve file
        scan_result = classify_and_flag(
            lc_file,
            min_size_kb=50,
            min_finite_rows=100,
            min_finite_fraction=0.50,
        )
        
        # Determine QC status
        status, details = determine_qc_status(scan_result)
        df.at[idx, "fitsQcStatus"] = status
        df.at[idx, "fitsQcReason"] = details
    
    # Save to new parquet
    df.to_parquet(output_path, index=False)
    print(f"\nQC results written to: {output_path}")
    
    # Print summary
    pass_count = (df["fitsQcStatus"] == "pass").sum()
    warning_count = (df["fitsQcStatus"] == "warning").sum()
    fail_count = (df["fitsQcStatus"] == "fail").sum()
    unknown_count = (df["fitsQcStatus"] == "unknown").sum()
    
    print(f"\nQC Summary:")
    print(f"  Pass:    {pass_count}")
    print(f"  Warning: {warning_count}")
    print(f"  Fail:    {fail_count}")
    print(f"  Unknown: {unknown_count}")
    print(f"  Total:   {len(df)}")


def main():
    parser = argparse.ArgumentParser(description="Scan a folder recursively and flag suspicious FITS files.")
    parser.add_argument("folder", help="Folder to scan recursively")
    parser.add_argument("--csv", dest="csv_path", default=None, help="Optional CSV output path")
    parser.add_argument("--min-size-kb", type=int, default=50, help="Flag files smaller than this size in KB")
    parser.add_argument("--min-finite-rows", type=int, default=100, help="Flag files with fewer finite time/flux rows than this")
    parser.add_argument("--min-finite-fraction", type=float, default=0.5, help="Flag files with lower finite-row fraction than this")
    args = parser.parse_args()

    folder = Path(args.folder)
    if not folder.exists() or not folder.is_dir():
        raise SystemExit(f"Folder does not exist or is not a directory: {folder}")

    results = scan_folder(
        folder=folder,
        min_size_kb=args.min_size_kb,
        min_finite_rows=args.min_finite_rows,
        min_finite_fraction=args.min_finite_fraction,
    )
    print_summary(results)

    if args.csv_path:
        csv_path = Path(args.csv_path)
        write_csv(results, csv_path)
        print(f"\nCSV report written to: {csv_path}")
    
    # Process TESSAugmented.parquet if it exists
    print("\n" + "="*80)
    print("Processing TESSAugmented.parquet for QC...")
    print("="*80)
    process_augmented_parquet(folder)


if __name__ == "__main__":
    main()

In [ ]:
# Optional FITS QC execution cell.
# This cell does nothing unless RUN_DATA_PIPELINE is changed to True.

if RUN_DATA_PIPELINE:
    from pathlib import Path
    process_augmented_parquet(Path(TESS_CACHE_FOLDER))
else:
    print("FITS QC execution skipped because RUN_DATA_PIPELINE is False.")

## 2.7 Dataset Coverage

The fallback strategy greatly increased usable TESS coverage.

| Pipeline stage | Approximate coverage |
|---|---:|
| Nearest TIC + SPOC only | 12.0% |
| Top-five TIC candidates + SPOC/QLP | 22.5% |
| Top-five TIC candidates + SPOC/QLP/TESSCut | 96.5% |

This result is central to the project. TESSCut was not a minor addition; it transformed the dataset from a small standard-product sample into a large multi-provenance sample suitable for class-level comparison.

In [ ]:
import pandas as pd

coverage_stages = pd.DataFrame([
    {"Stage": "Nearest TIC + SPOC only", "CoveragePercent": 12.0},
    {"Stage": "Top-5 candidates + SPOC/QLP", "CoveragePercent": 22.5},
    {"Stage": "Top-5 candidates + SPOC/QLP/TESSCut", "CoveragePercent": 96.51},
])

overall_provenance = pd.DataFrame([
    {"Provenance": "TESSCut", "Count": 5034, "Percent": 68.30},
    {"Provenance": "SPOC", "Count": 467, "Percent": 6.34},
    {"Provenance": "QLP", "Count": 1612, "Percent": 21.87},
    {"Provenance": "Missing", "Count": 257, "Percent": 3.49},
])

display(coverage_stages)
display(overall_provenance)

## 2.8 Provenance Distribution by Variable-Star Family

The final dataset is highly uneven across provenance categories. This matters because class-level model behavior must be interpreted in the context of which photometric products are available for each astrophysical population.

Several patterns are especially important:

- **RR Lyrae** objects are overwhelmingly represented through TESSCut.
- **Long-period variables** have strong QLP representation.
- **XRAY** objects are rare, making them difficult for any supervised model.
- Many families would be severely underrepresented without TESSCut.

This uneven provenance distribution motivates the later analysis of class-specific provenance sensitivity.

In [ ]:
family_stats = pd.DataFrame([
    {"Family": "CEPHEID", "TESSCut": 639, "SPOC": 29, "QLP": 168, "Missing": 0, "Total": 836},
    {"Family": "CV", "TESSCut": 525, "SPOC": 66, "QLP": 4, "Missing": 0, "Total": 595},
    {"Family": "DSCT_SXPHE", "TESSCut": 766, "SPOC": 34, "QLP": 153, "Missing": 0, "Total": 953},
    {"Family": "ECLIPSING", "TESSCut": 814, "SPOC": 8, "QLP": 174, "Missing": 0, "Total": 996},
    {"Family": "ELLIPSOIDAL_ROT", "TESSCut": 705, "SPOC": 122, "QLP": 170, "Missing": 0, "Total": 997},
    {"Family": "LONG_PERIOD", "TESSCut": 173, "SPOC": 90, "QLP": 717, "Missing": 0, "Total": 980},
    {"Family": "RRLYR", "TESSCut": 934, "SPOC": 3, "QLP": 39, "Missing": 0, "Total": 976},
    {"Family": "XRAY", "TESSCut": 20, "SPOC": 24, "QLP": 12, "Missing": 0, "Total": 56},
    {"Family": "YSO", "TESSCut": 458, "SPOC": 91, "QLP": 175, "Missing": 0, "Total": 724},
])

for col in ["TESSCut", "SPOC", "QLP", "Missing"]:
    family_stats[col + "_pct"] = (100 * family_stats[col] / family_stats["Total"]).round(2)

display(family_stats)

## 2.9 Chapter Summary

This chapter establishes the dataset foundation for the rest of the project.

The main conclusions are:

1. VSX provides astrophysical labels and sky positions.
2. Raw VSX types are grouped into broader, physically meaningful families.
3. VSX objects are cross-matched to TIC using a **5.0 arcsecond** radius.
4. Up to **five** TIC candidates are retained to improve availability while preserving match diagnostics for later analysis.
5. Light curves are acquired through a **SPOC → QLP → TESSCut** fallback hierarchy.
6. TESSCut dramatically increases dataset completeness.
7. FITS-level quality control identifies corrupted or suspicious files.
8. Detrending is not part of the data-pipeline phase and is evaluated later as a separate experiment.

The resulting dataset is appropriate for the next stage of the study: extracting physically interpretable light-curve features and evaluating how those features behave across photometric provenance categories.

Cross-match reliability will be treated later as part of the investigation into the TESSCut performance gap, rather than as part of the dataset-construction chapter.